 # **02. Household generation and conventional-load calculation (kWh) (+ plot generation)**

 This code first builds households according to a given criterion (shares of single-person households, couples with children, etc.) and selects, for each member of each household, a random profile from the dataset corresponding to that class of people.  
 It then associates each activity with a conventional electricity-consumption value (appliances and lighting, expressed as energy in kWh).  
 Finally, it prints summary plots on the allocation of conventional consumption across households in various formats.  
 NOTE: the building-occupancy profile of the different members is defined in the following file, together with the assignment of additional loads.

 ## 2.1 Import libraries

In [1]:
# ---------------------------- Standard library
import os
import glob
import re
import copy
import math
import random
import datetime
from datetime import timedelta
from itertools import chain
from collections import Counter, defaultdict

# ---------------------------- Scientific/data libraries
import numpy as np
import pandas as pd
from pandas import ExcelWriter
from tqdm import tqdm

# ---------------------------- Plotting libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm


# Get the start time
start_time = datetime.datetime.now()
print(f"Start time : {start_time}")


Start time : 2026-07-10 09:55:10.522341


 ## 2.2 Simulation parameters

In [2]:
# Define the number of households to simulate (WARNING: this value has to be aligned with the reference local energy system to be analized in the following modules)
num_households = 146

# Region(s) to consider when building households and calculating consumption
selected_region = 'Cen'                 # Choose among (N-O, N-E, Cen, Sud, Isl)

# Probabilities associated with household types (given 100 households, this is the probability that each belongs to a specific type).
household_probabilities = {
    'single': 0.38,                   # Single-person households
    'couple_wo_children': 0.21,       # Couples without children
    'couple_w_children_3': 0.15,      # Couples with 1 child (3 members)
    'couple_w_children_4': 0.12,      # Couples with 2 children (4 members)
    'couple_w_children_5': 0.03,      # Couples with 3 children (5 members)
    'couple_w_children_6': 0.01,      # Couples with 4 children (6 members)
    'single_parent_M_2': 0.05,        # Single fathers with 1 child (2 members)
    'single_parent_F_2': 0.05         # Single mothers with 1 child (2 members)
}


household_profiles_name = f'profiles_{selected_region}_{num_households}_households'
plot_name = f'{selected_region}_{num_households}_household_profiles'

# Set the withdrawal limit for individual meters/PODs of domestic users
cap_kw = 3.0


#------------------ For plot generation --------------------------------------------------------------------------------------
selected_season = 'summer'  

# Share of existing induction cooktops in the population
# (WARNING: be aware that if introduced an existing induction hob share, it has to be adjusted the assignment 
# of new induction hobs when computing additional loads in module 3; otherwise double counting for some households)
induction_share = 0                                 # value between 0 and 1, percentage of households with electric induction cooktops .
induction_share_pct = induction_share*100           # percentage value

# Filter only the appliances of interest for plotting (please refere to dictionary "appliance_english_dict" for English equivalents)
allowed_appliances = [
    # Keep this list aligned with the English appliance names used in
    # Database_appliances_ENG-ITA.xlsx / sheet "Power consumption".
    # "washing machine",
    # "dryer",
    "dishwasher",
    # "standby consumption",
    # "refrigerator",
    # "freezer compartment",
    # "steam mop",
    # "iron",
    # "food processor",
    # "blender",
    # "electric coffee grinder",
    # "slicer",
    # "bread maker",
    # "beverage heater",
    # "toaster",
    # "microwave oven",
    # "electric grill",
    # "electric cooktops",
    # "electric ovens",
    # "DVD player",
    # "VHS player",
    # "TV decoder",
    # "antenna",
    # "video game console",
    # "desktop PC",
    # "laptop",
    # "printer",
    # "mobile phone",
    # "landline phone",      
    # "answering machine",
    # "fax machine",
    # "Hi-Fi system",
    # "hair dryer",
    # "energy-efficient light bulb",
    # "traditional light bulb",
    # "vacuum cleaner",     
    # "cleaning robot",
    # "CRT TV",
    # "plasma TV",
    # "LCD TV",
    "LED TV"
    ]


 ## 2.3 Directories

In [3]:
# --------------
#   WARNING: adjust base directories to the one where this file.jpynb is located
# --------------

# Generic directories
input_folder = 'input'
output_folder = 'output'

#---------------------- Input/output paths for household activity-profile generation------------------

base_weekly_individual_profiles_dir = os.path.join(output_folder, '01_activity_profiles', 'weekly_profiles')
individual_profiles_input_dir = os.path.join(base_weekly_individual_profiles_dir, selected_region)
household_activity_profiles_output_folder = os.path.join(output_folder, '02_household_activity_profiles_Xls')


#---------------------- Input/output paths for appliance and lighting consumption calculation ------------------

household_profiles_input_dir = household_activity_profiles_output_folder
# household_profiles_input_dir = r"C:\\Users\\Francesco\\OneDrive - Politecnico di Milano\\Lavoro_FM_Online\\Dottorato FM\\elab_VSCode_PhD\\output\\02_household_activity_profiles_Xls"
consumption_input_folder = os.path.join(input_folder, '02_consumption')
consumption_output_xls_folder = os.path.join(output_folder, '02_base_consumption_Xls')


#---------------------- Paths for appliance-consumption plots ------------------------

# Path of the folder containing the files
output_pictures_folder = os.path.join(output_folder, '02_base_consumption_Pics')
appliance_pictures_output_folder = os.path.join(output_pictures_folder, 'appliance_profiles')

# Create the folders if they do not exist
for folder in [input_folder, 
                 output_folder, 
                 household_activity_profiles_output_folder, 
                 consumption_input_folder, 
                 consumption_output_xls_folder,
                 output_pictures_folder,
                 appliance_pictures_output_folder]:
    os.makedirs(folder, exist_ok=True)

 ## 2.4 Selectable sections

In [4]:
# Whether to build household activity profiles
RUN_CREATE_HOUSEHOLDS = True

# Whether to calculate traditional electricity consumption (lighting and appliances) associated with households
RUN_CONSUMPTION = True

# Whether to generate summary plots for appliance consumption
RUN_APPLIANCE_PLOTS = True

 ## 2.5 Dictionaries

### 2.5.1 Dictionaries for household activity profile and conventional loads

In [5]:
profile_sheets = {
    'winter': 'Winter profiles',
    'spring': 'Spring profiles',
    'summer': 'Summer profiles',
    'autumn': 'Autumn profiles',
    'final_winter_day': 'Final winter day'
}


#------------------ Consumption-calculation dictionaries
selectable_periods = ['winter', 'spring', 'summer', 'autumn']
season_abbr_dict = {'winter': 'Win', 
                    'spring': 'Spr', 
                    'summer': 'Sum',
                    'autumn': 'Aut'}

season_english_dict = {
    'winter': 'Winter', 
    'spring': 'Spring',
    'summer': 'Summer', 
    'autumn':'Autumn'
    }

order_types = ["single",
               "couple_wo_children",
               "single_parent_M_2",
               "single_parent_F_2",
               "couple_w_children_3",
               "couple_w_children_4",
               "couple_w_children_5",
               "couple_w_children_6"
                ]


# ------------ Dictionaries for appliance-consumption plots

day_order = ['wed','thu','fri','sat','sun','mon','tue']

# Ordered list of appliances (41 items). Source appliance names are intentionally kept in Italian because they match the input Excel database.
appliances_ordered = [
    "washing machine",
    "dryer",
    "dishwasher",
    "standby consumption",
    "refrigerator",
    "freezer compartment",
    "steam mop",
    "iron",
    "food processor",
    "blender",
    "electric coffee grinder",
    "slicer",
    "bread maker",
    "beverage heater",
    "toaster",
    "microwave oven",
    "electric grill",
    "electric cooktops",      # Currently, 6.7% of households in Italy have them; they should be excluded here
    "electric ovens",
    "DVD player",
    "VHS player",
    "TV decoder",
    "antenna",
    "video game console",
    "desktop PC",
    "laptop",
    "printer",
    "mobile phone",
    "landline phone",          
    "answering machine",
    "fax machine",
    "Hi-Fi system",
    "hair dryer",
    "energy-efficient light bulb",
    "traditional light bulb",
    "vacuum cleaner",         
    "cleaning robot",
    "CRT TV",
    "plasma TV",
    "LCD TV",
    "LED TV"
]


appliance_english_dict = {
    # Legacy Italian aliases kept only for backwards compatibility with older outputs.
    'Lavatrice': 'washing machine',
    'Asciugatrice': 'dryer',
    'Lavastoviglie': 'dishwasher',
    'Consumi_standby': 'standby consumption',
    'Frigorifero': 'refrigerator',
    'Vano_congelatore': 'freezer compartment',
    'Lavapavimenti a vapore': 'steam mop',
    'Ferro da stiro': 'iron',
    'Robot cucina': 'food processor',
    'Frullatore': 'blender',
    'Macina caffè elettrici': 'electric coffee grinder',
    'Affettatrice': 'slicer',
    'Macchina per pane': 'bread maker',
    'Scaldatore bevande': 'beverage heater',
    'Tostapane': 'toaster',
    'Forno microonde': 'microwave oven',
    'Griglia elettrica': 'electric grill',
    'piani_cottura_elettrici': 'electric cooktops',             # Currently, 6.7% of households in Italy have them; they should be excluded here
    'forni_elettrici': 'electric ovens',
    'Lettore DVD': 'DVD player',
    'Lettore VHS': 'VHS player',
    'Decoder TV': 'TV decoder',
    'Antenna': 'antenna',
    'Consolle per videogames': 'video game console',
    'PC desktop': 'desktop PC',
    'PC portatile': 'laptop',
    'Stampante': 'printer',
    'Telefono cellulare': 'mobile phone',
    'Telefono fisso': 'landline phone',
    'Segreteria': 'answering machine',
    'FAX': 'fax machine',
    'Impianto Hi-Fi': 'Hi-Fi system',
    'Asciugacapelli': 'hair dryer',
    'lampadina_ad_alta_efficienza': 'energy-efficient light bulb',
    'lampadina_tradizionali': 'traditional light bulb',
    'Aspirapolvere': 'vacuum cleaner',
    'Robot pulizia': 'cleaning robot',
    'TV a tubo catodico': 'CRT TV',
    'TV al plasma': 'plasma TV',
    'TV LCD': 'LCD TV',
    'TV LED': 'LED TV'
}


# Dictionaries for compatibility between older Italian-named dataframes (ISTAT) and English ones (e.g. EUROSTAT)
CYCLE_BASED_APPLIANCES = ['washing machine', 'dryer', 'dishwasher']
CONSTANT_LOAD_APPLIANCES = ['refrigerator', 'freezer compartment']
FIXED_CONSUMPTION_APPLIANCES = ['refrigerator', 'freezer compartment', 'standby consumption']
LIGHTING_USE_COLUMN = 'Artificial light use'

ACTIVITY_ALIASES = {
    'lighting': ['Illuminazione', 'Lighting'],
    'house_cleaning': ['Pulizia casa', 'House cleaning'],
    'watching_tv_pc': ['Guardare TV/PC', 'Watching TV/PC'],
}







# Define day categories
day_categories = ['working_day', 'saturday', 'sunday']
day_category_english_dict = {
    'working_day': 'Working day',
    'saturday': 'Saturday',
    'sunday': 'Sunday'
    }


color_by_household_type = {
    "single": "lightblue",      # or "#66c2a5"
    "couple_wo_children": "violet",             # or "#8da0cb"
    "single_parent_M_2": "saddlebrown",    # or "#e5c494"
    "single_parent_F_2": "gold",            # or "#b3b3b3"
    "couple_w_children_3": "orange",         # or "#fc8d62"
    "couple_w_children_4": "darkred",           # or "#e78ac3"
    "couple_w_children_5": "green",    # or "#a6d854"
    "couple_w_children_6": "lightgreen",           # or "#ffd92f"
}


### 2.5.2 Compatibility ITA/ENG dictionaries

In [6]:
#------------------ Compatibility dictionaries and helpers
# These helpers let ENG_02 read both the original Italian/legacy outputs and the
# new English outputs produced. Internally, names are normalized to the
# English/canonical convention used downstream by this module.

HOUSEHOLD_TYPE_ALIASES = {
    'single': 'single',
    'membro_isolato': 'single',

    'couple_wo_child': 'couple_wo_children',
    'couple_wo_children': 'couple_wo_children',
    'coppiaSF': 'couple_wo_children',

    'couple_w_child_3': 'couple_w_children_3',
    'couple_w_children_3': 'couple_w_children_3',
    'couple_w_child_4': 'couple_w_children_4',
    'couple_w_children_4': 'couple_w_children_4',
    'couple_w_child_5': 'couple_w_children_5',
    'couple_w_children_5': 'couple_w_children_5',
    'couple_w_child_6': 'couple_w_children_6',
    'couple_w_children_6': 'couple_w_children_6',

    'mono_M_2': 'single_parent_M_2',
    'single_parent_M_2': 'single_parent_M_2',
    'Mono_M_2': 'single_parent_M_2',
    'mono_F_2': 'single_parent_F_2',
    'single_parent_F_2': 'single_parent_F_2',
    'Mono_F_2': 'single_parent_F_2',
}

PROFILE_FILENAME_ALIASES = {
    'single': ['single', 'membro_isolato'],
    'couple_wo_child_head': ['couple_wo_child_head', 'coppiaSF_caponu', 'coppiaSF_caponucleo'],
    'couple_wo_child_spouse': ['couple_wo_child_spouse', 'coppiaSF_coniug', 'coppiaSF_coniuge'],
    'couple_w_child_head': ['couple_w_child_head', 'coppiaCF_caponucleo', 'coppiaCF_caponu', 'coppiaCF_capohousehold'],
    'couple_w_child_spouse': ['couple_w_child_spouse', 'coppiaCF_coniuge'],
    'couple_w_child_child': ['couple_w_child_child', 'coppiaCF_figlio'],
    'mono_M_father': ['mono_M_father', 'Mono_M_padre'],
    'mono_M_child': ['mono_M_child', 'Mono_M_figlio'],
    'mono_F_mother': ['mono_F_mother', 'Mono_F_madre'],
    'mono_F_child': ['mono_F_child', 'Mono_F_figlio'],
}

PROFILE_SHEET_ALIASES = {
    # ENG_01 converted/English outputs
    # Legacy full Italian names
    # Legacy abbreviated Italian names used by older files
    'winter': [        'Winter profiles', 'winter', 'Winter',        'Profili inverno', 'Inverno',        'Profili INV', 'INV'    ],
    'spring': [        'Spring profiles', 'spring', 'Spring',        'Profili primavera', 'Primavera',        'Profili PRI', 'PRI'    ],
    'summer': [        'Summer profiles', 'summer', 'Summer',        'Profili estate', 'Estate',        'Profili EST', 'EST'    ],
    'autumn': [        'Autumn profiles', 'autumn', 'Autumn',        'Profili autunno', 'Autunno',        'Profili AUT', 'AUT'    ],
    'final_winter_day': [        'Final winter day', 'final_winter_day', 'Final Winter Day',        'Giorno finale inverno', 'Ultimo giorno inverno', 'Profili giorno finale inverno',
            'Giorno finale INV', 'Ultimo giorno INV', 'Profili giorno finale INV'    ],
}

SEASON_ALIASES = {
    'winter': 'winter', 'inverno': 'winter', 'inv': 'winter',
    'spring': 'spring', 'primavera': 'spring', 'pri': 'spring',
    'summer': 'summer', 'estate': 'summer', 'est': 'summer',
    'autumn': 'autumn', 'fall': 'autumn', 'autunno': 'autumn', 'aut': 'autumn',
    'final_winter_day': 'final_winter_day',
    'ultimo_giorno_inverno': 'final_winter_day',
    'giorno_finale_inverno': 'final_winter_day',
    'giorno_finale_inv': 'final_winter_day',
    'ultimo_giorno_inv': 'final_winter_day',
}



























 ## 2.6 Functions

### 2.6.0 F(x) compatibility ITA/ENG input loading

In [7]:
# -------------------------------------------
#   Sec 2.6.0)  FUNCTIONS FOR ENABLING READING ORIGINAL ITALIAN/ENGLISH INPUTS 
# -------------------------------------------

# Function for return canonical household type label in English
def canonical_household_type(household_type):
    """Return the canonical English household-type label."""
    if household_type is None:
        return household_type
    household_type = str(household_type)
    if household_type in HOUSEHOLD_TYPE_ALIASES:
        return HOUSEHOLD_TYPE_ALIASES[household_type]

    # Normalize common legacy patterns such as coppiaCF_3 or coppiaSF_2.
    if household_type.startswith('couple_w_child_'):
        return household_type.replace('couple_w_child_', 'couple_w_children_')
    if household_type.startswith('couple_w_children_'):
        return household_type
    if household_type.startswith('mono_M_'):
        return household_type.replace('mono_M_', 'single_parent_M_')
    if household_type.startswith('mono_F_'):
        return household_type.replace('mono_F_', 'single_parent_F_')
    if household_type.startswith('single_parent_M_') or household_type.startswith('single_parent_F_'):
        return household_type

    return household_type


# Function to normalize probabilities assigned to households
def normalize_household_probabilities(probabilities):
    """Normalize household-type keys and check that probabilities are usable."""
    normalized = defaultdict(float)
    for household_type, probability in probabilities.items():
        normalized[canonical_household_type(household_type)] += probability

    total_probability = sum(normalized.values())
    if not np.isclose(total_probability, 1.0):
        raise ValueError(
            f"Household probabilities must sum to 1. Current sum: {total_probability}"
        )

    return dict(normalized)


# Function to normalize labels
def normalize_for_matching(label):
    """Normalize a label for robust sheet-name matching."""
    return re.sub(r'[^a-z0-9]+', '', str(label).casefold())


# Function to identify correct sheets depending on used key
def find_compatible_sheet(sheet_names, season_key):
    """Return the actual sheet name for a canonical season key."""
    aliases = PROFILE_SHEET_ALIASES.get(season_key, [])
    wanted = {normalize_for_matching(alias) for alias in aliases}
    for sheet_name in sheet_names:
        if normalize_for_matching(sheet_name) in wanted:
            return sheet_name
    return None


# Function to detect correct individual category based on label
def detect_profile_type_from_filename(file_name):
    """Detect the individual profile category from English or legacy Italian file names."""
    file_name_cf = str(file_name).casefold()
    for profile_type, aliases in PROFILE_FILENAME_ALIASES.items():
        if any(alias.casefold() in file_name_cf for alias in aliases):
            return profile_type
    return None


# Function to normalize column names for household profiles
def normalize_household_profile_column_name(column_name, default_season=None):
    """Normalize generated-household columns to Household_X_Member_Y_<season>_<type>."""
    if not isinstance(column_name, str) or column_name == 'Datetime':
        return column_name

    match = re.match(r'^(Household|Nucleo)_(\d+)_(Member|Membro)_(\d+)_(.+)$', column_name)
    if not match:
        return column_name

    household_number = match.group(2)
    member_number = match.group(4)
    tail = match.group(5)

    season_candidates = sorted(SEASON_ALIASES.keys(), key=len, reverse=True)
    for season_candidate in season_candidates:
        if tail == season_candidate or tail.startswith(season_candidate + '_'):
            season = SEASON_ALIASES[season_candidate]
            raw_household_type = tail[len(season_candidate):].lstrip('_')
            household_type = canonical_household_type(raw_household_type)
            return f"Household_{household_number}_Member_{member_number}_{season}_{household_type}"

    if default_season is not None:
        season = SEASON_ALIASES.get(str(default_season), str(default_season))
        household_type = canonical_household_type(tail)
        return f"Household_{household_number}_Member_{member_number}_{season}_{household_type}"

    return column_name


# Function to normalize columns in dataframe
def normalize_household_profile_columns(df, default_season=None):
    """Normalize all household-profile columns in a DataFrame."""
    df = df.copy()
    df.columns = [normalize_household_profile_column_name(c, default_season) for c in df.columns]
    return df


# Function to extract household type in a canonical way
def extract_household_type(household_name):
    """Extract and canonicalize the household type from a household identifier."""
    parts = str(household_name).split("_")
    raw_type = "_".join(parts[2:]) if len(parts) >= 3 and parts[0] == 'Household' else str(household_name)
    return canonical_household_type(raw_type)




# -------------------------------------------
#   Sec 2.6.0bis)  FUNCTIONS FOR ENABLING READING ORIGINAL ITALIAN/ENGLISH APPLIANCE INPUTS 
# -------------------------------------------


# Function to normalize labels between ITA and ENG dataframes
def normalize_label(label):
    """Normalize labels for robust ITA/ENG matching."""
    return re.sub(r'[^a-z0-9]+', '', str(label).casefold())

# Function to add aliases appliance use
def canonical_use_type(use_value):
    """Normalize appliance-use labels from Italian or English to internal keys."""
    aliases = {
        'constant': ['constant use', 'Uso costante'],
        'prolonged': ['prolonged use', 'Uso prolungato'],
        'punctual': ['punctual use', 'point use', 'Uso puntuale'],
    }
    normalized = normalize_label(use_value)
    for canonical, labels in aliases.items():
        if normalized in {normalize_label(label) for label in labels}:
            return canonical
    return str(use_value)


# Function to add aliases for appliance type
def canonical_appliance_type(appliance_type):
    """Normalize appliance-type labels from Italian or English to internal keys."""
    aliases = {
        'conventional': ['conventional', 'Convenzionale'],
        'exclusive': ['exclusive', 'Esclusivo'],
    }
    normalized = normalize_label(appliance_type)
    for canonical, labels in aliases.items():
        if normalized in {normalize_label(label) for label in labels}:
            return canonical
    return str(appliance_type)

# Function for return labels for comparison
def expand_activity_aliases(activity):
    """Return all known ITA/ENG labels for a requested activity."""
    activity_norm = normalize_label(activity)
    for aliases in ACTIVITY_ALIASES.values():
        if activity_norm in {normalize_label(alias) for alias in aliases}:
            return aliases
    return [activity]


# Function to check proper conversion of activity names
def activity_matches(info, accepted_activities):
    """Check whether an appliance activity matches any accepted ITA/ENG label."""
    if isinstance(accepted_activities, str):
        accepted_activities = expand_activity_aliases(accepted_activities)

    accepted = {normalize_label(activity) for activity in accepted_activities}
    candidates = [
        info.get('Activity'),
        info.get('Activity_ITA'),
        info.get('Activity_ENG'),
    ]
    return any(normalize_label(candidate) in accepted for candidate in candidates if pd.notna(candidate))


# Function to allow aliases in input file
def get_multiindex_column_values(df, appliance, unit_aliases):
    """Return non-null values from a two-level Excel header, allowing unit aliases."""
    for unit in unit_aliases:
        key = (appliance, unit)
        if key in df.columns:
            return df[key].dropna().values

    available_units = [second for first, second in df.columns if first == appliance]
    raise KeyError(
        f"No compatible unit column found for appliance '{appliance}'. "
        f"Accepted units: {unit_aliases}. Available units: {available_units}"
    )



### 2.6.1 F(x) Household activity profiles creation

In [8]:
# -------------------------------------------
#   Sec 2.6.1)  FUNCTIONS TO CREATE HOUSEHOLD ACTIVITY PROFILES 
# -------------------------------------------
if RUN_CREATE_HOUSEHOLDS:

    # Function to load activity-profile lists for the different classes of individuals in the selected region
    def build_profile_index(input_dir, profile_sheets):
        profiles_by_type = {t: {s: [] for s in profile_sheets} for t in [
            'single',
            'couple_wo_child_head','couple_wo_child_spouse',
            'couple_w_child_head','couple_w_child_spouse','couple_w_child_child',
            'mono_M_father','mono_M_child',
            'mono_F_mother','mono_F_child'
        ]}

        if not os.path.isdir(input_dir):
            raise FileNotFoundError(
                f"Individual weekly-profile folder not found: {input_dir}. "
                "Check selected_region and base_weekly_individual_profiles_dir."
            )

        for path in glob.glob(os.path.join(input_dir, '*.xlsx')):
            fname = os.path.basename(path)
            profile_type = detect_profile_type_from_filename(fname)
            if profile_type is None:
                print(f"Skipping unrecognized individual-profile file: {fname}")
                continue

            excel_file = pd.ExcelFile(path, engine='openpyxl')

            for season_key in profile_sheets:
                sheet_name = find_compatible_sheet(excel_file.sheet_names, season_key)
                if sheet_name is None:
                    print(
                        f"Warning: no compatible sheet for season '{season_key}' in {fname}. "
                        f"Available sheets: {excel_file.sheet_names}"
                    )
                    continue

                df = pd.read_excel(
                    path,
                    sheet_name=sheet_name,
                    engine='openpyxl',
                    index_col=0,
                    parse_dates=True
                )

                for col in df.columns:
                    profiles_by_type[profile_type][season_key].append(df[col])

        # Print a compact loading summary to make missing profile classes immediately visible.
        print('Loaded individual weekly profiles by type and season:')
        for profile_type, profiles_by_season in profiles_by_type.items():
            counts = {season: len(profile_list) for season, profile_list in profiles_by_season.items()}
            print(f'  {profile_type}: {counts}')



        return profiles_by_type
 
    # ------------------- OLD VERSION V01 (only ITA files as imput) -----------------
    # def build_profile_index(input_dir, profile_sheets):
    #     profiles_by_type = {t: {s: [] for s in profile_sheets} for t in [
    #         'single',
    #         'couple_wo_child_head','couple_wo_child_spouse',
    #         'couple_w_child_head','couple_w_child_spouse','couple_w_child_child',
    #         'mono_M_father','mono_M_child',
    #         'mono_F_mother','mono_F_child'
    #     ]}

    #     for path in glob.glob(os.path.join(input_dir, '*.xlsx')):
    #         fname = os.path.basename(path)
    #         # load all sheets 'Winter profiles', 'Spring profiles', ...
    #         sheets = pd.read_excel(
    #             path,
    #             sheet_name=list(profile_sheets.values()),
    #             engine='openpyxl',
    #             index_col=0,
    #             parse_dates=True
    #         )
    #         # CORRECT ITERATION: use profile_sheets.items() to get the season key
    #         for season_key, sheet_name in profile_sheets.items():
    #             df = sheets[sheet_name]
    #             for col in df.columns:
    #                 series = df[col]
    #                 if 'single' in fname or 'membro_isolato' in fname:
    #                     profiles_by_type['single'][season_key].append(series)
    #                 if 'couple_wo_child_head' in fname or 'coppiaSF_caponu' in fname:
    #                     profiles_by_type['couple_wo_child_head'][season_key].append(series)
    #                 if 'couple_wo_child_spouse' in fname or 'coppiaSF_coniug' in fname:
    #                     profiles_by_type['couple_wo_child_spouse'][season_key].append(series)
    #                 if 'couple_w_child_head' in fname or 'coppiaCF_capohousehold' in fname:
    #                     profiles_by_type['couple_w_child_head'][season_key].append(series)
    #                 if 'couple_w_child_spouse' in fname or 'coppiaCF_coniuge' in fname:
    #                     profiles_by_type['couple_w_child_spouse'][season_key].append(series)
    #                 if 'couple_w_child_child' in fname or 'coppiaCF_figlio' in fname:
    #                     profiles_by_type['couple_w_child_child'][season_key].append(series)
    #                 if 'mono_M_father' in fname or 'Mono_M_padre' in fname:
    #                     profiles_by_type['mono_M_father'][season_key].append(series)
    #                 if 'mono_M_child' in fname or 'Mono_M_figlio' in fname:
    #                     profiles_by_type['mono_M_child'][season_key].append(series)
    #                 if 'mono_F_mother' in fname or 'Mono_F_madre' in fname:
    #                     profiles_by_type['mono_F_mother'][season_key].append(series)
    #                 if 'mono_F_child' in fname or 'Mono_F_figlio' in fname:
    #                     profiles_by_type['mono_F_child'][season_key].append(series)
    #    return profiles_by_type



    # Efficient household-generation function
    def create_consistent_households(
        num_households, prob_households, profile_sheets, profiles_by_type
    ):
        household_type_options, weights = zip(*prob_households.items())
        selected_types = np.random.choice(household_type_options, size=num_households, p=weights)

        # 1) For each household, decide the list of sources and the index
        selections = {}
        for i, household_type in enumerate(selected_types, 1):
            household_type = canonical_household_type(household_type)
            # parse the type
            if household_type == 'single':
                members = ['single']
            elif household_type == 'couple_wo_children':
                members = ['couple_wo_child_head','couple_wo_child_spouse']
            elif household_type.startswith('couple_w_children'):
                total = int(household_type.split('_')[-1])
                children_count = total - 2
                members = ['couple_w_child_head','couple_w_child_spouse'] + ['couple_w_child_child']*children_count
            elif household_type.startswith('single_parent_M'):
                total = int(household_type.split('_')[-1])
                children_count = total - 1
                members = ['mono_M_father'] + ['mono_M_child']*children_count
            else:  # single_parent_F
                total = int(household_type.split('_')[-1])
                children_count = total - 1
                members = ['mono_F_mother'] + ['mono_F_child']*children_count

            # Choose the index only once in the winter season
            member_choices = []
            for src in members:
                pool = profiles_by_type[src]['winter']
                if not pool:
                    available_counts = {
                        profile_type: {season: len(profile_list) for season, profile_list in profiles_by_season.items()}
                        for profile_type, profiles_by_season in profiles_by_type.items()
                    }
                    raise ValueError(
                        f"No profile for {src} in winter. "
                        "Check that the corresponding ENG_01 weekly-profile file exists and that its sheet names are recognized. "
                        f"Loaded profile counts: {available_counts}"
                    )
                idx = np.random.randint(len(pool))
                member_choices.append((src, idx))
            selections[i] = (household_type, member_choices)

        # 2) Recompose the profiles season by season
        results = {}
        for season in profile_sheets:
            all_series = []
            for i, (household_type, member_choices) in selections.items():
                for j, (src, idx) in enumerate(member_choices, 1):
                    pool = profiles_by_type[src][season]
                    if idx >= len(pool):
                        raise IndexError(f"Index {idx} out of range per {src} in {season}")
                    s = pool[idx].copy()
                    s.name = f"Household_{i}_Member_{j}_{season}_{household_type}"
                    all_series.append(s)

            # final concatenation and one reset
            df = pd.concat(all_series, axis=1)\
                .reset_index()\
                .rename(columns={'index':'Datetime'})

            # sort columns naturally
            cols = [c for c in df.columns if c!='Datetime']
            def key(c):
                m = re.match(r"Household_(\d+)_Member_(\d+)", c)
                return (int(m.group(1)), int(m.group(2))) if m else (float('inf'),)
            ordered_columns = ['Datetime'] + sorted(cols, key=key)
            results[season] = df[ordered_columns]

        return results, selected_types

#---------------------------------------------------------------------------------------------------------------------------------------------------

### 2.6.2 F(x) Conventional load calculation

In [9]:
# -------------------------------------------
#   Sec 2.6.2)  FUNCTIONS TO CALCULATE ELECTRICAL LOADS (APPLIANCES AND LIGHTING) 
# -------------------------------------------

if RUN_CONSUMPTION:
    
    ### (2nd CALLED FUNCTION) ###
    # Function to select a random value from a stable-consumption column
    def select_random_consumption_class(appliance, df):
        """
        Select a random consumption value for the appliance.
        Cycle-based appliances use kWh/cycle; constant-load appliances use kWh/year.
        """
        if appliance in CYCLE_BASED_APPLIANCES:
            values = get_multiindex_column_values(
                df,
                appliance,
                ['[kWh/cycle]', '[kWh/cicle]', '[kWh/ciclo]']
            )
        elif appliance in CONSTANT_LOAD_APPLIANCES:
            values = get_multiindex_column_values(
                df,
                appliance,
                ['[kWh/y]']
            )
        else:
            print(f"No consumption rule available for {appliance}.")
            return 0

        values = [value for value in values if pd.notna(value)]
        if len(values) == 0:
            print(f"No value available for {appliance}.")
            return 0

        return random.choice(values)
    # ----- OLD VERSION V01
    # def select_random_consumption_class(appliance, df):
    #     """
    #     Select a random value from the appliance column in the DataFrame.
    #     For cycle-based appliances, use the '[kWh/cycle]' column.
    #     For appliances with annual consumption, use the '[kWh/y]' column.
    #     """
    #     column_data = df[appliance].dropna()
    #     if not column_data.empty:
    #         if appliance in ['washing machine', 'dryer', 'dishwasher']:
    #             random_value = df[(appliance, '[kWh/cicle]')].dropna().sample(n=1).values[0]
    #         # elif appliance in ['refrigerator', 'freezer compartment']:
    #         #     random_value = df[(appliance, '[kWh/y]')].dropna().sample(n=1).values[0]
    #         elif appliance in ['refrigerator', 'freezer compartment']:
    #             # take the array of available values
    #             arr = df[(appliance, '[kWh/y]')].dropna().values
    #             if len(arr)>0:
    #                 random_value = random.choice(arr)
    #             else:
    #                 random_value = 0
    #         return random_value
    #     else:
    #         print(f"No value available for {appliance}.")
    #         return 0


    ### (1st CALLED FUNCTION) ###
    # Function to select the power for appliance categories for which only annual consumption is known
    def get_appliance_power(appliance_id, energy_consumption_df, stable_consumption_df, appliances_df):
        """
        Get the power, or the consumption per cycle, of the appliance.
        """
        if appliance_id in CYCLE_BASED_APPLIANCES:
            return select_random_consumption_class(appliance_id, energy_consumption_df)
        elif appliance_id in CONSTANT_LOAD_APPLIANCES:
            return select_random_consumption_class(appliance_id, stable_consumption_df) / (365 * 24) * 1000
        return 0
    # def get_appliance_power(appliance_id, energy_consumption_df, stable_consumption_df, appliances_df):
    #     """
    #     Get the power (or, in some cases, the consumption per cycle) of the appliance
    #     based on consumption data and any required conversions.
    #     """
    #     if appliance_id in ['washing machine', 'dryer', 'dishwasher']:
    #         return select_random_consumption_class(appliance_id, energy_consumption_df)
    #     elif appliance_id in ['refrigerator', 'freezer compartment']:
    #         return select_random_consumption_class(appliance_id, stable_consumption_df) / (365 * 24) * 1000


    ### (5th CALLED FUNCTION) ###
    # Function to select an exclusive appliance based on the activity (one TV, one light bulb, one vacuum cleaner only)
    def select_exclusive_appliance(appliance_info, activity):
        """
        Select an exclusive appliance for the specified activity.
        The requested activity can be passed either in Italian or English.
        """
        accepted_activities = expand_activity_aliases(activity)

        filtered_appliances = {
            appliance_item: info
            for appliance_item, info in appliance_info.items()
            if canonical_appliance_type(info.get('Appliance_type')) == 'exclusive'
            and activity_matches(info, accepted_activities)
        }
        
        if not filtered_appliances:
            available = sorted({
                f"{info.get('Activity_ITA')} / {info.get('Activity_ENG')}"
                for info in appliance_info.values()
                if canonical_appliance_type(info.get('Appliance_type')) == 'exclusive'
            })
            raise ValueError(
                f"No exclusive appliance found for activity {activity}. "
                f"Available exclusive activities: {available}"
            )
        
        appliances = list(filtered_appliances.keys())
        ownership_probabilities = [
            filtered_appliances[appliance_id]['Ownership_probability']
            for appliance_id in appliances
        ]
        return random.choices(appliances, weights=ownership_probabilities, k=1)[0]



    ### (4th CALLED FUNCTION - 2nd inside "determine_appliance_ownership_by_household") ###
    # Function to associate the different parameters with the appliances
    def create_custom_appliance_info(appliances_df, energy_consumption_df, stable_consumption_df, household_id_arg=None):
        """
        Create a customized information dictionary for each appliance.
        Appliance names are read in English from the converted Excel file.
        Activity labels are stored in both Italian and English; the Italian label is kept
        as the primary label because the household activity profiles are ISTAT-based.
        """
        info_dict = {}
        for index, row in appliances_df.iterrows():
            appliance = row['Appliance']
            power = row['Power [W]']
            power_choice = row['Power choice']
            
            if str(power_choice).casefold().startswith('see energy consumption'):
                power_value = get_appliance_power(
                    appliance,
                    energy_consumption_df,
                    stable_consumption_df,
                    appliances_df
                )
            else:
                power_value = float(power)

            activity_ita = row['Activity (ITA)']
            activity_eng = row['Activity (ENG)']

            info_dict[appliance] = {
                'Power': power_value,
                'Usage_probability': row['Probability of use'],
                'Ownership_probability': row['Ownership probability'],
                'Use': canonical_use_type(row['Type of use']),
                'Activity_number': row['Activity number'],
                'Activity': activity_ita,                               # primary activity label: ISTAT/Italian
                'Activity_ITA': activity_ita,
                'Activity_ENG': activity_eng,
                'Cycle_duration': row['Duration of the cycle [minutes]'],
                'Appliance_type': canonical_appliance_type(row['Type of appliance'])
            }
        return info_dict
    # def create_custom_appliance_info(appliances_df, energy_consumption_df, stable_consumption_df, household_id_arg=None):
    #     """
    #     Create a customized information dictionary for each appliance.
    #     """
    #     info_dict = {}
    #     for index, row in appliances_df.iterrows():
    #         appliance = row['Appliance']
    #         power = row['Power [W]']
    #         power_choice = row['Power choice']
            
    #         if power_choice.startswith('see energy consumption'):
    #             power_value = get_appliance_power(appliance, energy_consumption_df, stable_consumption_df, appliances_df)
    #         else:
    #             power_value = float(power)
    #         # if household_id is not None and appliance in ['refrigerator','Vano_congelatore']:
    #         #     print(f"[DEBUG] household={household_id:<12}  appliance={appliance:<15}  power={power_value:.3f}")

    #         info_dict[appliance] = {
    #             'Power': power_value,
    #             'Usage_probability': row['Probability of use'],
    #             'Ownership_probability': row['Ownership probability'],
    #             'Use': row['Type of use'],  # e.g., point, constant, or prolonged use
    #             'Activity_number': row['Activity number'],
    #             'Activity': row['Activity (ITA)'],                          # Using Italian activity classification to refer to ISTAT dataset
    #             'Cycle_duration': row['Duration of the cycle [minutes]'],
    #             'Appliance_type': row['Type of appliance']
    #         }
    #     return info_dict





    ### (CALLED FUNCTION - 1st inside "calculate_consumption") ###
    # Function to identify the household of each individual in the activity DataFrame
    def identify_households(activity_df):
        """
        Identify households and household types from activity DataFrame column names.
        Supports canonical English columns and common legacy Italian variants after
        normalizing them to the canonical format.
        """
        normalized_columns = [normalize_household_profile_column_name(c) for c in activity_df.columns]
        extracted = pd.Series(normalized_columns, index=activity_df.columns).str.extract(
            r'(Household_\d+)_Member_\d+_(?:winter|spring|summer|autumn|final_winter_day)_(.*)'
        )
        extracted.columns = ['Household', 'Household_type']
        if extracted.isnull().any().any():
            bad_columns = list(activity_df.columns[extracted.isnull().any(axis=1)])
            raise ValueError(
                "Invalid column format in at least one column. Examples: "
                f"{bad_columns[:5]}"
            )
        extracted['Household_type'] = extracted['Household_type'].map(canonical_household_type)
        return extracted['Household'], extracted['Household_type']
    # def identify_households(activity_df):
    #     """
    #     Identify households and household types from the activity DataFrame column names.
    #     The expected format is: "Household_(number)_Member_(number)_(season)_(household type)".
        
    #     Return two Series: one for the household (e.g., "Household_1") and one for the household type.
    #     """
    #     extracted = activity_df.columns.to_series().str.extract(r'(Household_\d+)_Member_\d+_[^_]+_(.*)')
    #     extracted.columns = ['Household', 'Household_type']
    #     if extracted.isnull().any().any():
    #         raise ValueError("Invalid column format in at least one column.")
    #     return extracted['Household'], extracted['Household_type']


    ### (3rd CALLED FUNCTION - 1st inside "determine_appliance_ownership_by_household") ###
    # Function to define appliance ownership among individuals (not everyone owns the same appliances), assigning different energy classes to them
    def determine_appliance_ownership_by_household(activity_df, appliances_df, energy_consumption_df, stable_consumption_df):
        """
        Determine appliance ownership for each household.
        
        Returns:
        - appliance_ownership_by_person: dictionary with the list of owned appliances for each person
        - custom_appliances_by_person: dictionary with customized appliance information for each person
        - num_households: number of distinct households
        """
        households, household_types = identify_households(activity_df)
        household_association = households + "_" + household_types  # e.g. "Household_1_couple_w_children_4"
        household_names = household_association.unique()
        
        
        # take all possible kWh/y classes
        refrigerator_values = stable_consumption_df[('refrigerator','[kWh/y]')].dropna().tolist()
        freezer_values  = stable_consumption_df[('freezer compartment','[kWh/y]')].dropna().tolist()


        # for each household, choose the random class only once
        refrigerators_by_household = {
            household_id: random.choice(refrigerator_values)/(365*24)*1000  # W
            for household_id in households
        }
        freezers_by_household  = {
            household_id: random.choice(freezer_values)/(365*24)*1000  # W
            for household_id in households
        }
        

        # save the final results here
        appliance_ownership_by_household = {}
        custom_appliances_by_household = {}
        
        print(f"Number of identified households: {len(household_names)}")
        
        # For each household (identified by the "Household_X" part)
        for household in sorted(set(households)):
            

            # 1) Create the base dictionary for ALL appliances
            base_info = create_custom_appliance_info(
                            appliances_df,
                            energy_consumption_df,
                            stable_consumption_df)
            
            # 2) Inject the selected fixed power for refrigerator/freezer
            base_info['refrigerator']['Power']      = refrigerators_by_household[household]
            base_info['freezer compartment']['Power'] = freezers_by_household[household]
            
            # 3) Save it under the simple household key
            custom_appliances_by_household[household] = base_info

            # 4) Build the ownership list
            ownership = [
                select_exclusive_appliance(base_info, 'Illuminazione'),             # referred to Italian classification provided by ISTAT
                select_exclusive_appliance(base_info, 'Pulizia casa'),              # referred to Italian classification provided by ISTAT
                select_exclusive_appliance(base_info, 'Guardare TV/PC')             # referred to Italian classification provided by ISTAT
            ]

            # Conventional appliances
            for appliance_item, info in base_info.items():
                if info['Appliance_type'] == 'conventional' and (
                info['Ownership_probability'] == 1.0
                or random.random() <= info['Ownership_probability']
                ):
                    ownership.append(appliance_item)

            appliance_ownership_by_household[household] = ownership

        # Propagate to household members
        appliance_ownership_by_person = {}
        custom_appliances_by_person = {}
        for person in activity_df.columns:
            simple_household_id = re.match(r'(Household_\d+)', person).group(1)
            appliance_ownership_by_person[person] = appliance_ownership_by_household[simple_household_id]
            custom_appliances_by_person[person]   = custom_appliances_by_household[simple_household_id]

        
        return appliance_ownership_by_person, custom_appliances_by_person, len(household_names), household_names


    ### (9th CALLED FUNCTION - 2nd inside "calculate_consumption") ###
    # Function for the random selection of an appliance usage profile (e.g., washing machine, dishwasher)
    def select_random_profile(appliance, profiles_df):
        """
        Randomly select a usage profile for the appliance.
        """
        profile_columns = [col for col in profiles_df.columns if appliance.lower() in col.lower()]
        if not profile_columns:
            raise ValueError(f"No profile found for appliance: {appliance}")
        selected_column = random.choice(profile_columns)
        return profiles_df[selected_column]



    ### (6th CALLED FUNCTION) ###
    # Function to select a random usage profile and build a DataFrame with only the selected profiles
    def select_single_random_profile(df):
        """
        Select a single random profile for the dryer.
        """
        # # Split columns by appliance
        # washing_machine_columns = [
        #     'Lavatrice - A - standard', 
        #     'Lavatrice - A - delicato',
        #     'Lavatrice - B - standard', 
        #     'Lavatrice - B - delicato'
        # ]
        
        # dishwasher_columns = [
        #     'Lavastoviglie - standard', 
        #     'Lavastoviglie (ciclo ECO)'
        # ]
        dryer_columns = [
            'dryer _ 1', 
            'dryer _ 2', 
            'dryer _ 3', 
            'dryer _ 4'
        ]
        
        # Select a random profile for each appliance
        # washing_machine_profile = random.choice(washing_machine_columns)
        # dishwasher_profile = random.choice(dishwasher_columns)
        dryer_profile = random.choice(dryer_columns)
        
        # # # Create a new DataFrame with the selected profiles
        # # selected_profiles_df = df[[washing_machine_profile, dishwasher_profile, dryer_profile]]
        # selected_profiles_df = df[[washing_machine_profile, dishwasher_profile, dryer_profile]].copy()
        selected_profiles_df = df[[dryer_profile]].copy()
        
        # Rename the column for the dryer
        selected_profiles_df.rename(columns={dryer_profile: 'dryer'}, inplace=True)
        
        return selected_profiles_df





    ### (7th CALLED FUNCTION) ###
    # MAIN FUNCTION TO CALCULATE CONSUMPTION OF THE DIFFERENT APPLIANCES DURING THE YEAR

    def calculate_consumption(activity_df, custom_appliances, appliance_ownership, ten_minute_df, appliance_profiles_df, selected_appliance_profiles_df):
        """
        1) Identify households and parameters
            2) Preallocate empty structures
            3) Calculate constant consumption (baseload)
            3bis) Map customized information by household
            4) Main loop: variable consumption + remaining cap
            5) Compose fixed, variable, and complete dictionaries
        """
        
        # ---------------- Part 1: Household identification ----------------
        households, household_types = identify_households(activity_df)
        household_association = households + "_" + household_types
        household_names = household_association.unique()
        
        
        # ---------------- Part 2: Parameters and preallocation ----------------
        dryer_delay_steps = 6   
        STEP_MIN   = 10   
        # MAX_CONSUMPTION_PER_HOUSEHOLD = 0.5 # if 'electric cooktops' in household_appliances[household] else 0.5
        
        # Mapping from the simple key (e.g., "Household_1") to the full key ("Household_1_couple_w_children_4")
        full_household_mapping = {}
        for col in activity_df.columns:
            simple = re.match(r'(Household_\d+)', col).group(1)
            full_household_mapping[simple] = household_association[col]
        
        n_timesteps = len(activity_df)
        consumption_df = pd.DataFrame(0, index=activity_df.index, columns=activity_df.columns)
        variable_appliance_consumption_df = pd.DataFrame(0, index=activity_df.index, columns=[])
        constant_consumption_df = pd.DataFrame(0, index=activity_df.index, columns=household_names)
        
        # dictionary for fixed consumption only by household
        fixed_appliances_by_household = {
            household_id: pd.DataFrame(0, index=activity_df.index, columns=[]) 
            for household_id in household_names
        }
        
        # ---------------- Part 3: Baseload calculation (constant uses) -------------
        primary_household_members = [key for key in appliance_ownership.keys() if re.match(r'Household_\d+_Member_1', key)]
        
        for member1  in primary_household_members:
            household_number = re.match(r'Household_(\d+)_Member_1', member1 ).group(1)
            full_household_id = full_household_mapping[f'Household_{household_number}']

            # Get the CUSTOMIZED information dictionary for that household
            household_info = custom_appliances[member1 ]
            household_appliance_list = appliance_ownership[member1 ]
        
        
            # Identify appliances with constant use.
            fixed_appliances = [e for e in household_appliance_list if household_info[e]['Use']=='constant']
            for appliance_id in fixed_appliances:
                # calculate kWh (energia) per timestep
                power_kw = household_info[appliance_id]['Power'] / 1000
                kwh  = power_kw * (STEP_MIN / 60)
        
                # Fill the full fixed-consumption column for this appliance.
                fixed_appliances_by_household[full_household_id][appliance_id] = kwh
        
                # add it to the overall baseload for that household across ALL timesteps
                constant_consumption_df[full_household_id] += kwh


        # Array containing existing consumption (energy kWh): constant-appliance baseload and appliances already running from previous timesteps
        variable_consumption_array = np.zeros((n_timesteps, num_households), dtype=np.float64)
        remaining_capacity_debug = np.zeros_like(variable_consumption_array)
        # ongoing_debug = np.zeros_like(consumption_variable_array)
        household_to_idx = {household: idx for idx, household in enumerate(household_names)}
        
        
        
        # --- Part 3: Preallocate appliance consumption (energy kWh per timestep) by household ---
        # Build a mapping for each household's owned appliances. For example, take the ownership list
        # of the first person encountered for each household.
        household_appliances = {}
        for person, appliances in appliance_ownership.items():
            simple = re.match(r'(Household_\d+)', person).group(1)
            full   = full_household_mapping[simple]
            household_appliances.setdefault(full, appliances)
            
        # Preallocate, for each household, a NumPy array for the appliances
        appliance_consumption_arrays = {
            n: np.zeros((n_timesteps, len(household_appliances[n]))) 
            for n in household_names
        }
        appliance_to_idx = {
            n: {app:i for i,app in enumerate(household_appliances[n])} 
            for n in household_names
        }



        #    build the dictionary of customized caps by household
        max_consumption_per_household = {}
        for household_id in household_names:
            apps = household_appliances.get(household_id, [])
            if 'electric cooktops' in apps:
                # If the household has an electric cooktop, increase the cap to 1 kWh/step.
                max_consumption_per_household[household_id] = 1
            else:
                # Default cap.
                max_consumption_per_household[household_id] = 0.5


        
        # Initialize the remaining time for each appliance/person and the dryer delay counter
        remaining_time = {person: {appliance_item: 0 for appliance_item in appliance_ownership[person]} for person in activity_df.columns}
        appliance_profiles = {}
        
        delay_counter = {
            person: {appliance_id: 0 for appliance_id in appliance_ownership[person]}
            for person in activity_df.columns
        }    
            
        # Activities for which lighting is excluded
        excluded_activities = [1, 7, 8, 11]

        
        # Prepare profiles for appliances with prolonged use
        for person in appliance_ownership:
            appliance_profiles[person] = {}
            for appliance_item in appliance_ownership[person]:
                if appliance_item.lower() in [app.lower() for app in CYCLE_BASED_APPLIANCES]:
                    if appliance_item.lower() == "dryer":
                        appliance_profiles[person][appliance_item] = selected_appliance_profiles_df['dryer']
                    else:
                        appliance_profiles[person][appliance_item] = select_random_profile(appliance_item, appliance_profiles_df)
                else:
                    appliance_profiles[person][appliance_item] = None
        
        
        # ------------------------------------------------------
        # 4) Main loop: consumption uso prolonged e point
        # ------------------------------------------------------
        # Initialize prolonged-use consumption that was already active at (t-1)
        existing_consumption_array = np.zeros((n_timesteps, num_households))
        
        for person in tqdm(activity_df.columns, desc=f"Calculating consumption for {num_households} households"):
            household = household_association[person]
            appliance_info_for_person = custom_appliances[person]
            col_idx = household_to_idx[household]
            cap = max_consumption_per_household[household]
            
            
            for i, activity in activity_df[person].items():
                row_idx = activity_df.index.get_loc(i)
                
                # calculate constant consumption (base, energy kWh) only once
                base    = constant_consumption_df.at[i, household]
                
                for appliance_item in appliance_ownership[person]:
                    info = appliance_info_for_person[appliance_item]
                    consumption = 0
                    
                    # 4.1) Manage prolonged use
                    if info['Use'] == 'prolonged':

                        # 4.1.1) SPECIAL CASE: dryer with delay always applied
                        if appliance_item == 'dryer':   
                            # calculate remaining steps
                            ts_idx          = consumption_df.index.get_loc(i)
                            total_steps     = len(consumption_df)
                            steps_remaining = total_steps - ts_idx

                            # start the cycle only if this is the first step of activity 9 and no cycle is already in progress
                            if activity == info['Activity_number'] and (info['Usage_probability'] == 1.0 or random.random() <= info['Usage_probability']) and remaining_time[person][appliance_item] == 0:
                                # set the remaining time to the full duration (shifted by the delay)
                                appliance_profiles[person][appliance_item] = selected_appliance_profiles_df['dryer']
                                remaining_time[person][appliance_item] = info['Cycle_duration']
                                # record how many steps still need to be skipped
                                delay_counter[person][appliance_item] = dryer_delay_steps
                                
                            # if a delay is still active, decrement it and do not consume
                            if delay_counter[person][appliance_item] > 0:
                                delay_counter[person][appliance_item] -= 1
                                consumption = 0

                            # otherwise, if the delays have expired and there is still remaining time
                            elif remaining_time[person][appliance_item] > 0:
                                # if the end of the simulation would be exceeded, skip
                                if steps_remaining <= 0:
                                    consumption = 0
                                else:
                                    # step-by-step consumption
                                    idx      = (info['Cycle_duration'] - remaining_time[person][appliance_item]) // STEP_MIN
                                    consumption  = info['Power'] * appliance_profiles[person][appliance_item].iloc[int(idx)]
                                    remaining_time[person][appliance_item] -= STEP_MIN
                            else:
                                consumption = 0
                
                        else:
                            # 4.1.2) START CYCLE (without delay) and choose the profile only once
                            if remaining_time[person][appliance_item] == 0 \
                            and activity == info['Activity_number'] \
                            and (info['Usage_probability'] == 1.0 or random.random() <= info['Usage_probability']):
                                # select the random profile only here
                                prof = select_random_profile(appliance_item, appliance_profiles_df)
                                appliance_profiles[person][appliance_item] = prof
                                # initialize the cycle duration
                                remaining_time[person][appliance_item] = info['Cycle_duration']
                                # first consumption (index 0)
                                consumption = info['Power'] * prof.iloc[0]
                                remaining_time[person][appliance_item] -= STEP_MIN

                            # b) CYCLE IN PROGRESS: step-by-step consumption from the same profile
                            elif remaining_time[person][appliance_item] > 0:
                                prof = appliance_profiles[person][appliance_item]
                                idx  = int((info['Cycle_duration'] - remaining_time[person][appliance_item]) // STEP_MIN)
                                consumption = info['Power'] * prof.iloc[idx]
                                remaining_time[person][appliance_item] -= STEP_MIN
                        
                            # c) OTHERWISE: no consumption
                            else:
                                consumption = 0
                        
                        # 4.1.3) If there is consumption, update the DataFrames
                        if consumption > 0:
                            consumption_df.loc[i, person] += consumption
                            if appliance_item not in variable_appliance_consumption_df.columns:
                                variable_appliance_consumption_df[appliance_item] = 0

                        existing_consumption_array[row_idx, col_idx] += consumption
                        variable_consumption_array[row_idx, col_idx] += consumption
                        
                        if appliance_item in appliance_to_idx[household]:
                            idx_el = appliance_to_idx[household][appliance_item]
                            appliance_consumption_arrays[household][row_idx, idx_el] += consumption
                        continue
                            
                    # 4.2) Manage lighting
                    elif activity_matches(info, ACTIVITY_ALIASES['lighting']):
                        if ten_minute_df.loc[i, LIGHTING_USE_COLUMN] and activity not in excluded_activities:
                            power_kw = info['Power'] / 1000
                            consumption = power_kw * (10 * 60) / 3600  # Consumption over 10 minutes (energy, kWh)

                    
                    # 4.3) Manage point use
                    elif info['Activity_number'] == activity and info['Use'] == 'punctual':
                        if info['Usage_probability'] == 1.0 or random.random() <= info['Usage_probability']:
                            power_kw = info['Power'] / 1000
                            consumption = power_kw * (10 * 60) / 3600
                            # if elettro not in df_consumption_appliances_variab.columns:
                            #     df_consumption_appliances_variab[elettro] = 0
            
                    
                    # 4.4) RECALCULATE THE REMAINING CAPACITY here, BEFORE blocking point consumption
                    ongoing = existing_consumption_array[row_idx, col_idx]
                    # residuo = MAX_CONSUMO_PER_NUCLEO - base - ongoing
                    remaining_capacity = cap - base - ongoing
                    remaining_capacity_debug[row_idx, col_idx] = remaining_capacity
                    # # (optional) also save the debug for ongoing consumption
                    # ongoing_debug[row_idx, col_idx] = ongoing

                    # Apply cap
                    if consumption>remaining_capacity:
                        consumption=0
                    
                    # if appliance use is confirmed, update the counts
                    if consumption>0:
                        consumption_df.at[i, person] += consumption
                        variable_consumption_array[row_idx, col_idx] += consumption
                        # update appliance array
                        idx_el = appliance_to_idx[household].get(appliance_item)
                        if idx_el is not None:
                            appliance_consumption_arrays[household][row_idx, idx_el] += consumption


        # Aggregated DataFrames by household (variable consumption)
        household_variable_consumption_df = pd.DataFrame(variable_consumption_array,
                                            index=activity_df.index,
                                            columns=household_names)    
        # totals = constant + variable consumption (energy, kWh)
        household_total_consumption_df = constant_consumption_df.add(household_variable_consumption_df, fill_value=0)


        
        # ---------------- Part 5: Dictionary composition -------------------
        variable_appliances_by_household = {}
        complete_appliances_by_household     = {}
        
        for household in household_names:
            # variable
            variable_df = pd.DataFrame(
                appliance_consumption_arrays[household],
                index=activity_df.index,
                columns=household_appliances[household]
            )
            variable_appliances_by_household[household] = variable_df

            # complete = fixed + variable, without duplicates
            df_all = pd.concat(
                [ fixed_appliances_by_household[household], variable_df ],
                axis=1
            )
            df_all = df_all.loc[:, ~df_all.columns.duplicated()]
            complete_appliances_by_household[household] = df_all   

        return (
            consumption_df,
            household_variable_consumption_df,
            household_total_consumption_df,
            variable_appliance_consumption_df,
            fixed_appliances_by_household,
            variable_appliances_by_household,
            complete_appliances_by_household,
            remaining_capacity_debug,
            constant_consumption_df,
            max_consumption_per_household
            # MAX_CONSUMPTION_PER_HOUSEHOLD
        )

    #---------------------------------------------------------------------------------------------------------------------------------------------------

### 2.6.3 F(x) Plotting conventional loads (appliances)

In [10]:
# -------------------------------------------
#   Sec 2.6.3)  FUNCTIONS TO PLOT CONVENTIONAL LOADS 
# -------------------------------------------

if RUN_APPLIANCE_PLOTS:
    # Function to reorder i households secondo un dato criterio (es. prima i "single", poi i "Coppia_SF" ecc)
    def get_type_order(household):
        t = extract_household_type(household)
        for idx, typ in enumerate(order_types):
            if typ in t:
                return idx
        return 100  # se non trovato, va in fondo
    
    
    # Function to define how many colors to associate with each household type based on how many households of that type exist
    def get_gradation_colors(base_hex, n, v_min=0.8, v_max=1.0):
        """
        Restituisce una lista di n colori in gradazioni a partire da base_hex.
        L'intervallo della luminosità (Value in HSV) varia linearmente tra v_min e v_max.
        
        Parametri:
        - base_hex: stringa esadecimale del colore base, es. "#fc8d62"
        - n: numero di gradazioni richieste
        - v_min: minimum value per il canale Value (tra 0 e 1)
        - v_max: maximum value per il canale Value (tra 0 e 1)
        """
        # Convert il colore di base in RGB (tuple di float in [0,1])
        base_rgb = mcolors.to_rgb(base_hex)
        # Convert l'RGB in HSV (Hue, Saturation, Value)
        base_hsv = mcolors.rgb_to_hsv(base_rgb)
        
        # Generate n linearly distributed values tra v_min e v_max
        # In this way, all colors have the same hue and saturation and vary only in the "value" channel
        values = np.linspace(v_min, v_max, n)
            
        colors = []
        for v in values:
            new_hsv = (base_hsv[0], base_hsv[1], v)
            new_rgb = mcolors.hsv_to_rgb(new_hsv)
            colors.append(mcolors.to_hex(new_rgb))
        return colors
    
    #---------------------------------------------------------------------------------------------------------------------------------------------------

 ## 2.7 Create households by selecting individual profiles

In [11]:
# ------------------------------------------
#   Sec 2.7 -- 1.0) Input data normalization/adaptation (ITA/ENG)
# ------------------------------------------
household_probabilities = normalize_household_probabilities(household_probabilities)


# ------------------------------------------





# ------------------------------------------
#   Sec 2.7 -- 2.0) Household activity profile definition
# ------------------------------------------
if RUN_CREATE_HOUSEHOLDS:
    # preload into memory
    profiles_by_type = build_profile_index(individual_profiles_input_dir, profile_sheets)

    # Create the requested number of households without repeated Excel I/O.
    households_by_season, selected_household_types = create_consistent_households(
        num_households,
        prob_households=household_probabilities,
        profile_sheets=profile_sheets,
        profiles_by_type=profiles_by_type#,
        #file_figlio=file_figlio
    )

    # build the summary
    counts_by_type = Counter(selected_household_types)
    summary_df = pd.DataFrame.from_records([
        {
            'Household type': household_type,
            'Count': counts_by_type[household_type],
            'Percentage (%)': round(100 * counts_by_type[household_type] / num_households, 2)
        }
        for household_type in household_probabilities.keys()
    ])

    # Save the generated households to a multi-sheet Excel file.
    out_file = os.path.join(household_activity_profiles_output_folder, f"profiles_{selected_region}_{num_households}_households.xlsx")
    with ExcelWriter(out_file) as writer:
        for season, df in households_by_season.items():
            df.to_excel(writer, sheet_name=season, index=False)
        # add the summary
        summary_df.to_excel(writer, sheet_name='Summary', index=False)

Loaded individual weekly profiles by type and season:
  single: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  couple_wo_child_head: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  couple_wo_child_spouse: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  couple_w_child_head: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  couple_w_child_spouse: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  couple_w_child_child: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  mono_M_father: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  mono_M_child: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_day': 200}
  mono_F_mother: {'winter': 200, 'spring': 200, 'summer': 200, 'autumn': 200, 'final_winter_d

 ## 2.8 Calculate electrical loads from appliances and lighting

 The reported values are expressed as consumed energy (kWh) at each 10-minute timestep

In [12]:
if RUN_CONSUMPTION:

    # --------------------------------
    #   Sec 2.8 -- 1.0) Load input data and prepare profiles
    # --------------------------------

    # Start by reading input data and create the lighting DataFrame
    db = os.path.join(consumption_input_folder, 'Database_appliances_ENG-ITA.xlsx')
    energy_consumption_df  = pd.read_excel(db, sheet_name='Energy consumption', header=[0,1])
    stable_consumption_df  = pd.read_excel(db, sheet_name='Constant consumption', header=[0,1])
    lighting_schedule_df      = pd.read_excel(db, sheet_name='Lighting profile')
    appliances_df = pd.read_excel(db, sheet_name='Power consumption')
    appliance_profiles_df = pd.read_excel(db, sheet_name='Appliance cycle profiles')                    

    # Modify the ownership probability in the correct sheet
    mask = appliances_df["Appliance"] == "electric cooktops"
    appliances_df.loc[mask, "Ownership probability"] = induction_share

    # Print the ownership probability assigned to electric cooktops as a check.
    val = appliances_df.loc[mask, "Ownership probability"].values[0]
    print("Ownership probability for electric cooktops =", val)

    # lighting profile at 10-minute resolution
    lighting_schedule_df['data'] = pd.to_datetime(lighting_schedule_df['data'])
    detailed_rows = []
    for _, row in lighting_schedule_df.iterrows():
        for i in range(6):
            detailed_rows.append({
                'Datetime': row['data'] + pd.Timedelta(minutes=10*i),
                LIGHTING_USE_COLUMN: row[LIGHTING_USE_COLUMN]
            })

    ten_minute_df = pd.DataFrame(detailed_rows).set_index('Datetime')

    # Prepare appliance profiles (index: 'Minuti')
    appliance_profiles_df.set_index('Minutes', inplace=True)
    selected_appliance_profiles_df = select_single_random_profile(appliance_profiles_df)



    # --------------------------------
    #   Sec 2.8 -- 2.0) Assign appliance ownership to households 
    #                   (starting from the first season, WINTER)
    # --------------------------------

    first_season = selectable_periods[0]
    initial_activity_df = pd.read_excel(
        os.path.join(household_profiles_input_dir, f"{household_profiles_name}.xlsx"),
        sheet_name=first_season
    )
    
    initial_activity_df = normalize_household_profile_columns(
        initial_activity_df,
        default_season=first_season
    )
    initial_activity_df['Datetime'] = pd.to_datetime(initial_activity_df['Datetime'])
    initial_activity_df.set_index('Datetime', inplace=True)

    # Determine appliance ownership and customized appliance information only once.
    fixed_ownership, fixed_info, identified_household_count, household_names = \
        determine_appliance_ownership_by_household(
            initial_activity_df,
            appliances_df,
            energy_consumption_df,
            stable_consumption_df
        )

    if identified_household_count != num_households:
        raise ValueError(
            f"Expected {num_households} households, but identified {identified_household_count}."
        )



    # --------------------------------
    #   Sec 2.8 -- 3.0) Result containers
    # --------------------------------

    # Result containers for output Excel files and downstream summaries.
    individual_consumption_dict = {}                  # individual-person consumption time series
    variable_appliance_consumption_dict = {}          # variable consumption by appliance, timestep
    total_household_consumption_dict = {}             # total consumption by household, timestep
    appliance_summary_list = []                       # weekly appliance summary list
    household_summary_list = []                       # weekly household summary list
    aggregated_household_ts_dict = {}                 # aggregated household time series
    fixed_household_ts_dict = {}                      # fixed time series by household
    household_appliance_ts_dict = {}                  # appliance time series aggregated across households
    remaining_capacity_debug_dict = {}                # remaining-capacity debug DataFrames by season
    nested_appliances = {}



    # --------------------------------
    #   Sec 2.8 -- 4.0) Seasonal consumption loop
    # --------------------------------

    for season in selectable_periods:
        abbr = season_abbr_dict[season]
        season_file = os.path.join(
            household_profiles_input_dir,
            f"{household_profiles_name}.xlsx"
        )
        
        # 4.1) Load and normalize household activity profiles for the current season.
        activity_df = pd.read_excel(season_file, sheet_name=season)
        activity_df = normalize_household_profile_columns(
            activity_df,
            default_season=season
        )
        activity_df['Datetime'] = pd.to_datetime(activity_df['Datetime'])
        activity_df.set_index('Datetime', inplace=True)
        
        # 4.2) Reuse the appliance ownership and appliance information selected from the first season.
        local_ownership = {}
        local_info = {}

        for person in activity_df.columns:
            base_member_key = re.match(r'(Household_\d+_Member_\d+)', person).group(1)
            reference_key = next(k for k in fixed_ownership if k.startswith(base_member_key))
            local_info[person] = copy.deepcopy(fixed_info[reference_key])
            local_ownership[person] = fixed_ownership[reference_key]
        
        # 4.3) Calculate appliance and lighting consumption for the current season.
        (
            consumption_df,
            variable_households_df,
            total_households_df,
            variable_appliance_df,
            fixed_appliances_by_household,
            variable_appliances_by_household,
            appliances_by_household,
            remaining_capacity_debug,
            constant_consumption_df,
            max_consumption_per_household,
        ) = calculate_consumption(
            activity_df,
            local_info,
            local_ownership,
            ten_minute_df,
            appliance_profiles_df,
            selected_appliance_profiles_df
        )
        
        # 4.4) Store household-level appliance time series and debug outputs for later summaries.
        remaining_capacity_debug_dict[season] = pd.DataFrame(
            remaining_capacity_debug,
            index=activity_df.index,
            columns=household_names
        )

        nested_appliances[season] = appliances_by_household.copy()

        all_appliances_df = pd.concat(
            appliances_by_household.values(),
            axis=1
        ).fillna(0)

        household_appliance_ts_dict[season] = all_appliances_df.groupby(
            level=0,
            axis=1
        ).sum()
        
        # 4.5) Aggregate individual consumption and fixed-appliance consumption at household level.
        household_map = {
            person: re.match(r'(Household_\d+)_', person).group(1)
            for person in consumption_df.columns
        }

        household_individual_df = consumption_df.groupby(household_map, axis=1).sum()
        
        fixed_list = []

        for full_household_id, household_appliance_df in appliances_by_household.items():
            simple_household_id = re.match(r'(Household_\d+)', full_household_id).group(1)

            fixed_columns = [
                c for c in household_appliance_df.columns
                if c in FIXED_CONSUMPTION_APPLIANCES
            ]

            fixed_series = household_appliance_df[fixed_columns].sum(axis=1).rename(
                simple_household_id
            )
            fixed_list.append(fixed_series)

        fixed_household_df = pd.concat(fixed_list, axis=1).fillna(0)
        
        aggregated_household_ts_dict[season] = household_individual_df.add(
            fixed_household_df,
            fill_value=0
        )
        fixed_household_ts_dict[season] = fixed_household_df
        
        # 4.6) Store seasonal consumption time series.
        individual_consumption_dict[season] = consumption_df
        variable_appliance_consumption_dict[season] = variable_appliance_df
        total_household_consumption_dict[season] = total_households_df
        
        # 4.7) Build daily/weekly appliance-consumption summaries.
        df_all = household_appliance_ts_dict[season]

        daily_appliance_df = df_all.resample('D').sum()
        weekly_appliance_sum = daily_appliance_df.sum()

        daily_appliance_df['day'] = daily_appliance_df.index.dayofweek.map({
            0: 'mon',
            1: 'tue',
            2: 'wed',
            3: 'thu',
            4: 'fri',
            5: 'sat',
            6: 'sun'
        })

        appliance_by_day_df = daily_appliance_df.groupby('day').sum().reindex(day_order)

        summary_index = [f"{day} {abbr}" for day in day_order] + [
            f"week {abbr} [kWh_el]"
        ]

        appliance_week_df = pd.DataFrame(
            index=summary_index,
            columns=df_all.columns
        )

        for day_index, day in enumerate(day_order):
            appliance_week_df.iloc[day_index] = appliance_by_day_df.loc[day]

        appliance_week_df.iloc[-1] = weekly_appliance_sum.values
        appliance_summary_list.append(appliance_week_df)
    
        # 4.8) Build daily/weekly household-consumption summaries.
        daily_household_df = total_households_df.resample('D').sum()
        weekly_household_sum = daily_household_df.sum()

        daily_household_df['day'] = daily_household_df.index.dayofweek.map({
            0: 'mon',
            1: 'tue',
            2: 'wed',
            3: 'thu',
            4: 'fri',
            5: 'sat',
            6: 'sun'
        })

        household_by_day_df = daily_household_df.groupby('day').sum().reindex(day_order)

        household_week_df = pd.DataFrame(
            index=summary_index,
            columns=total_households_df.columns
        )

        for day_index, day in enumerate(day_order):
            household_week_df.iloc[day_index] = household_by_day_df.loc[day]

        household_week_df.iloc[-1] = weekly_household_sum.values
        household_summary_list.append(household_week_df)
        
        # Optional debug export for remaining capacity can be re-enabled here if needed.
        # remaining_capacity_df = pd.DataFrame(
        #     remaining_capacity_debug,
        #     index=activity_df.index,
        #     columns=household_names
        # )
        # remaining_capacity_df.to_excel(os.path.join(
        #     consumption_output_xls_folder,
        #     f'debug_remaining_consumable_capacity_{num_households}_households_{selected_region}_{season}.xlsx'
        # ))



    # --------------------------------
    #   Sec 2.8 -- 5.0) Appliance-level seasonal and annual statistics
    # --------------------------------

    # weekly_sum[(appliance, household, season)] contains the weekly kWh values.
    weekly_sum = {}

    for season, household_appliance_dict in nested_appliances.items():
        for household, household_appliance_df in household_appliance_dict.items():
            for appliance_item in household_appliance_df.columns:
                weekly_values = household_appliance_df[appliance_item].resample('W').sum()
                weekly_values = weekly_values[weekly_values > 0]
                weekly_sum[(appliance_item, household, season)] = weekly_values
    
    # Convert weekly values to seasonal values by multiplying each representative week by 13.
    season_sum = {}

    for key, weekly_values in weekly_sum.items():
        season_sum[key] = weekly_values.sum() * 13
    
    # Annual value = four 13-week seasons + 1/7 of the winter representative week.
    annual_sum = {}

    for (appliance_id, household_id, season), seasonal_kwh in season_sum.items():
        annual_sum.setdefault((appliance_id, household_id), 0.0)
        annual_sum[(appliance_id, household_id)] += seasonal_kwh
    
    for (appliance_id, household_id), annual_value in list(annual_sum.items()):
        winter_week_sum = weekly_sum.get(
            (appliance_id, household_id, 'winter'),
            pd.Series(dtype=float)
        ).sum()

        annual_sum[(appliance_id, household_id)] = annual_value + winter_week_sum / 7
    
    # Long-format table: one row per appliance-household pair.
    rows = []

    for (appliance_id, household_id), annual_value in annual_sum.items():
        row = {
            'Appliance': appliance_id,
            'Household': household_id,
            'annual_kWh': annual_value
        }

        for season in selectable_periods:
            row[f'{season}_season_kWh'] = season_sum.get(
                (appliance_id, household_id, season),
                0.0
            )

        rows.append(row)
    
    df_long = pd.DataFrame(rows)
    df_long = df_long.sort_values(
        ['Appliance', 'Household'],
        ascending=[True, True]
    )
    
    numeric_columns = [
        c for c in df_long.columns
        if c not in ['Appliance', 'Household']
    ]
    df_long[numeric_columns] = df_long[numeric_columns].apply(
        pd.to_numeric,
        errors='coerce'
    )
    
    # Appliance statistics across households: min, mean, max, and sum.
    aggregation_dict = {
        'annual_kWh': ['min', 'mean', 'max'],
        **{
            f'{season}_season_kWh': ['min', 'mean', 'max']
            for season in selectable_periods
        }
    }

    df_stats = df_long.groupby('Appliance').agg(aggregation_dict)
    df_stats.columns.names = ['Period', 'Statistic']
    
    period_label_map = {'annual_kWh': 'Year'}
    period_label_map.update({
        f'{season}_season_kWh': f'{season}_season'
        for season in selectable_periods
    })

    df_stats.columns = pd.MultiIndex.from_tuples(
        [
            (period_label_map.get(period, period), statistic)
            for period, statistic in df_stats.columns
        ],
        names=['Period', 'Statistic']
    )
    
    totals = df_long.groupby('Appliance').sum(numeric_only=True)[
        ['annual_kWh'] + [f'{season}_season_kWh' for season in selectable_periods]
    ]
    
    for source_column, period_label in period_label_map.items():
        df_stats[(period_label, 'sum')] = totals[source_column]
    
    ordered_stat_columns = []

    for period_label in ['Year'] + [f'{season}_season' for season in selectable_periods]:
        for statistic in ['min', 'mean', 'max', 'sum']:
            if (period_label, statistic) in df_stats.columns:
                ordered_stat_columns.append((period_label, statistic))

    df_stats_final = df_stats[ordered_stat_columns]



    # --------------------------------
    #   Sec 2.8 -- 6.0) Household-level seasonal and annual statistics
    # --------------------------------

    # Rename simple household columns (e.g., Household_1) using full names that include the household type.
    simple_to_full_household = {
        f"{fn.split('_')[0]}_{fn.split('_')[1]}": fn
        for fn in household_names
    }

    for season, df in aggregated_household_ts_dict.items():
        aggregated_household_ts_dict[season] = df.rename(columns=simple_to_full_household)

    seasons = selectable_periods
    full_household_names = aggregated_household_ts_dict['winter'].columns

    season_row_labels = {
        season: f"{season_english_dict[season]} season [kWh_el]"
        for season in seasons
    }
    
    df_seasonal = pd.DataFrame(
        index=list(season_row_labels.values()) + ['Annual consumption [kWh_el]'],
        columns=full_household_names,
        dtype=float
    )
    
    # For each season, calculate average weekly consumption and scale it to 13 weeks.
    mean_weekly = {}

    for season in seasons:
        weekly = aggregated_household_ts_dict[season].resample('W').sum()
        weekly_mean = weekly.mean(axis=0)
        mean_weekly[season] = weekly_mean
        df_seasonal.loc[season_row_labels[season]] = weekly_mean * 13
    
    # Annual consumption = four 13-week seasons + 1/7 of the winter representative week.
    annual = sum(mean_weekly[season] * 13 for season in seasons) + mean_weekly['winter'] * (1/7)
    df_seasonal.loc['Annual consumption [kWh_el]'] = annual
    
    # Seasonal and annual household-consumption statistics by household category.
    all_categories = sorted({extract_household_type(fn) for fn in household_names})
    annual_series = df_seasonal.loc['Annual consumption [kWh_el]']

    stats = ['mean', 'min', 'max']
    period_labels = ['Year'] + [f'{season}_season' for season in seasons]
    columns = pd.MultiIndex.from_product(
        [period_labels, stats],
        names=['Period', 'Statistic']
    )

    category_stats_df = pd.DataFrame(
        index=all_categories,
        columns=columns,
        dtype=object
    )
    
    for season in seasons:
        grouped_season = df_seasonal.loc[season_row_labels[season]].groupby(
            df_seasonal.columns.to_series().apply(extract_household_type)
        )

        category_stats_df[(f'{season}_season', 'mean')] = grouped_season.mean().reindex(all_categories)
        category_stats_df[(f'{season}_season', 'min')] = grouped_season.min().reindex(all_categories)
        category_stats_df[(f'{season}_season', 'max')] = grouped_season.max().reindex(all_categories)
    
    grouped_annual = annual_series.groupby(
        annual_series.index.to_series().apply(extract_household_type)
    )

    category_stats_df[('Year', 'mean')] = grouped_annual.mean().reindex(all_categories)
    category_stats_df[('Year', 'min')] = grouped_annual.min().reindex(all_categories)
    category_stats_df[('Year', 'max')] = grouped_annual.max().reindex(all_categories)
    
    category_stats_df = category_stats_df.fillna('n/d')

    category_stats_df = category_stats_df[
        [('Year', statistic) for statistic in stats]
        + [(f'{season}_season', statistic) for season in seasons for statistic in stats]
    ]



    # --------------------------------
    #   Sec 2.8 -- 7.0) Load-duration curves and cap-exceedance analysis
    # --------------------------------

    # Analyze load-duration curves and cap exceedance.
    
    # Define the cap in kW and convert it to kWh per 10-minute timestep.
    cap_kwh = cap_kw * (10/60)   # e.g. 3 kW → 0.5 kWh per timestep
    
    # Prepare two dictionaries: duration curves and exceedance summaries
    duration_sheets = {}
    over_sheets = {}
    
    # Define the desired household-category order.
    category_order = [
        'single',
        'couple_wo_children',
        'couple_w_children_3',
        'couple_w_children_4',
        'couple_w_children_5',
        'couple_w_children_6',
        'single_parent_M_2',
        'single_parent_F_2'
    ]
    
    for season in selectable_periods:
        season_df = aggregated_household_ts_dict[season]
        start = season_df.index.min()

        # Extract the representative week: from `start` to `start + 7 days`.
        week_df = season_df.loc[
            start : start + timedelta(days=7) - timedelta(seconds=1)
        ]
    
        # Load-duration curve: sort each household column in descending order.
        duration_df = week_df.apply(
            lambda col: col.sort_values(ascending=False).reset_index(drop=True)
        )

        # For each category in the chosen order, take all columns belonging to that category.
        column_to_category = duration_df.columns.to_series().apply(extract_household_type)

        ordered_cols = list(chain.from_iterable(
            sorted(duration_df.columns[column_to_category == cat])
            for cat in category_order
        ))

        # If there are columns with unexpected categories, append them at the end.
        extras = [c for c in duration_df.columns if c not in ordered_cols]
        ordered_cols += sorted(extras)
        
        duration_df = duration_df[ordered_cols]
        duration_sheets[season] = duration_df
        
        # Calculate how many timesteps exceed the cap and the percentage of the total.
        over_counts = (week_df > cap_kwh).sum()
        over_perc = over_counts / week_df.shape[0] * 100

        over_sheets[season] = pd.DataFrame({
            'n_timesteps_>cap': over_counts,
            '%_time_>cap': over_perc.round(2)
        })



    # --------------------------------
    #   Sec 2.8 -- 8.0) Save output DataFrames
    # --------------------------------

    # 8.1) Consumption time series (energy kWh) for INDIVIDUAL PERSONS (one sheet per season)
    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_ts_kWh_indiv_base.xlsx'
        ),
        engine='openpyxl'
    ) as w:
        for season, df in individual_consumption_dict.items():
            df.to_excel(w, sheet_name=season)
            
    # 8.2) Consumption time series (energy kWh) aggregated by HOUSEHOLD (one sheet per season)
    # Time series for TOTAL and FIXED consumption by household
    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_ts_kWh_aggregated_households_base.xlsx'
        ),
        engine='openpyxl'
    ) as w:
        for season in selectable_periods:
            # sheet with total consumption
            df_tot = aggregated_household_ts_dict[season]
            df_tot.to_excel(w, sheet_name=f"{season} TOT")

            # sheet with fixed consumption only
            df_fix = fixed_household_ts_dict[season]
            df_fix.to_excel(w, sheet_name=f"{season} FIX")

    # 8.3) Appliance time series aggregated across all households (energy kWh)
    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_ts_kWh_appliances_by_household.xlsx'
        ),
        engine='openpyxl'
    ) as w:
        for season, df in household_appliance_ts_dict.items():
            df.to_excel(w, sheet_name=season)

    # 8.4) Daily/weekly APPLIANCE summary (kWh and %)
    df_kwh_ele = pd.concat(appliance_summary_list)
    df_pct_ele = df_kwh_ele.div(df_kwh_ele.sum(axis=1), axis=0) * 100

    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_summary_appliances.xlsx'
        ),
        engine='openpyxl'
    ) as w:
        df_kwh_ele.to_excel(w, sheet_name='kWh el')
        df_pct_ele.to_excel(w, sheet_name='percentages')
    
    # 8.5) Daily/weekly HOUSEHOLD summary (kWh and %)
    df_kwh_nuc = pd.concat(household_summary_list)
    df_pct_nuc = df_kwh_nuc.div(df_kwh_nuc.sum(axis=1), axis=0) * 100

    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_summary_households_base.xlsx'
        ),
        engine='openpyxl'
    ) as w:
        df_kwh_nuc.to_excel(w, sheet_name='kWh el')
        df_pct_nuc.to_excel(w, sheet_name='percentages')

        # Sheet with seasonal consumption and the annual total by household.
        df_seasonal.to_excel(w, sheet_name='seasonal_and_annual')

        # Sheet with mean/min/max annual consumption by category.
        category_stats_df.to_excel(w, sheet_name='category_stats')
    
    # 8.6) Daily/weekly percentage consumption summary by HOUSEHOLD CATEGORY
    sheets = {'by_day': {}, 'by_week': {}}

    for household_week_df in household_summary_list:
        tag = household_week_df.index[0].split()[-1]
        cats = {col: extract_household_type(col) for col in household_week_df.columns}

        df_cat = household_week_df.groupby(cats, axis=1).sum()
        pct = df_cat.div(df_cat.sum(axis=1), axis=0) * 100

        days = [r for r in pct.index if not r.startswith('week')]
        week_sum = [r for r in pct.index if r.startswith('week')]

        sheets['by_day'][tag] = pct.loc[days]
        sheets['by_week'][tag] = pct.loc[week_sum]
    
    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_percentuali_category_household_base.xlsx'
        ),
        engine='openpyxl'
    ) as w:
        for sheet_name, by_season in sheets.items():
            pd.concat(by_season).to_excel(w, sheet_name=sheet_name)
            
    # 8.7) Load-duration curves for individual households
    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_load_curves_households_base.xlsx'
        ),
        engine='openpyxl'
    ) as writer:
        for season in selectable_periods:
            duration_sheets[season].to_excel(
                writer,
                sheet_name=f'duration_{season}',
                index_label='rank'
            )
            over_sheets[season].to_excel(
                writer,
                sheet_name=f'over_cap_{season}'
            )

    # 8.8) Average consumption of each appliance across households
    with pd.ExcelWriter(
        os.path.join(
            consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_cons_appl_medio_summary.xlsx'
        ),
        engine='openpyxl'
    ) as writer:
        # Sheet with all detailed information.
        df_long.to_excel(writer, sheet_name='by_household', index=False)

        # Sheet with max-min-mean values across seasons and the year.
        df_stats_final.to_excel(writer, sheet_name='Appliance_consumption')

    # 8.9) Evolution of remaining household consumption capacity
    debug_capacity_path = os.path.join(
        consumption_output_xls_folder,
        f'{num_households}_households_{selected_region}_debug_remaining_consumable_capacity.xlsx'
    )

    with pd.ExcelWriter(debug_capacity_path, engine='openpyxl') as writer:
        for season, remaining_capacity_df in remaining_capacity_debug_dict.items():
            remaining_capacity_df.to_excel(writer, sheet_name=season)


Ownership probability for electric cooktops = 0.0
Number of identified households: 146


Calculating consumption for 146 households:   0%|          | 0/330 [00:00<?, ?it/s]C:\Users\Francesco\AppData\Local\Temp\ipykernel_13496\380053883.py:634: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.006698333333333332' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  consumption_df.at[i, person] += consumption
Calculating consumption for 146 households:   0%|          | 1/330 [00:01<06:21,  1.16s/it]C:\Users\Francesco\AppData\Local\Temp\ipykernel_13496\380053883.py:634: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.31666666666666665' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  consumption_df.at[i, person] += consumption
Calculating consumption for 146 households:   1%|          | 2/330 [00:02<06:17,  1.15s/it]C:\Users\Francesco\AppDa

 ## 2.9 Print base-consumption plots

 Here I print selected plots of base electricity consumption only (appliances)

In [13]:
# %% [markdown]
#  ## 2.9 Generate base-consumption plots
# 
#  This section generates selected plots for base electricity consumption from conventional appliances and lighting.

# %%
if RUN_APPLIANCE_PLOTS:

    # --------------------------------
    #   Sec 2.9 -- 1.0) Load plotting input data
    # --------------------------------

    # Household daily/weekly consumption summary.
    household_summary_path = os.path.join(
        consumption_output_xls_folder,
        f'{num_households}_households_{selected_region}_summary_households_base.xlsx'
    )

    household_kwh_df = pd.read_excel(
        household_summary_path,
        sheet_name='kWh el',
        index_col=0
    )
    household_percentage_df = pd.read_excel(
        household_summary_path,
        sheet_name='percentages',
        index_col=0
    )

    # Household-category percentage summary.
    household_category_percentage_path = os.path.join(
        consumption_output_xls_folder,
        f'{num_households}_households_{selected_region}_percentuali_category_household_base.xlsx'
    )

    aggregated_household_categories_df = pd.read_excel(
        household_category_percentage_path,
        sheet_name='by_day',
        index_col=[0, 1]
    )

    # Appliance time series for the selected season.
    appliance_timeseries_path = os.path.join(
        consumption_output_xls_folder,
        f'{num_households}_households_{selected_region}_ts_kWh_appliances_by_household.xlsx'
    )

    appliance_timeseries_df = pd.read_excel(
        appliance_timeseries_path,
        sheet_name=selected_season,
        index_col=0,
        parse_dates=True
    )

    # Appliance daily/weekly consumption summary.
    appliance_summary_path = os.path.join(
        consumption_output_xls_folder,
        f'{num_households}_households_{selected_region}_summary_appliances.xlsx'
    )

    appliance_kwh_df = pd.read_excel(
        appliance_summary_path,
        sheet_name='kWh el',
        index_col=0
    )
    appliance_percentage_df = pd.read_excel(
        appliance_summary_path,
        sheet_name='percentages',
        index_col=0
    )

    # Appliance annual-consumption dispersion summary.
    appliance_dispersion_path = os.path.join(
        consumption_output_xls_folder,
        f'{num_households}_households_{selected_region}_cons_appl_medio_summary.xlsx'
    )

    appliance_dispersion_by_household_df = pd.read_excel(
        appliance_dispersion_path,
        sheet_name='by_household',
        index_col=0
    )



    # --------------------------------
    #   Sec 2.9 -- 2.0) Filter the selected season and order daily rows
    # --------------------------------

    # Build the selected-season row pattern, e.g. 'wed Sum'.
    selected_season_abbr = season_abbr_dict[selected_season]
    selected_season_pattern = '|'.join(
        f'^{day} {selected_season_abbr}$'
        for day in day_order
    )
    ordered_day_labels = [
        f'{day} {selected_season_abbr}'
        for day in day_order
    ]

    # Select only daily household rows for the selected season.
    df_season_households = household_kwh_df[
        household_kwh_df.index.str.match(selected_season_pattern)
    ].reindex(ordered_day_labels)

    # Select only daily appliance-percentage rows for the selected season.
    season_appliance_percentage_df = appliance_percentage_df[
        appliance_percentage_df.index.str.match(selected_season_pattern)
    ].reindex(ordered_day_labels)

    # Select only daily household-category rows for the selected season.
    # The source DataFrame has a MultiIndex; the second level contains labels such as 'wed Sum'.
    level_1_labels = pd.Series(
        aggregated_household_categories_df.index.get_level_values(1),
        index=aggregated_household_categories_df.index
    )
    selected_season_mask = level_1_labels.str.match(selected_season_pattern).fillna(False)
    df_season_households_aggregated = aggregated_household_categories_df.loc[
        selected_season_mask
    ].reindex(ordered_day_labels, level=1)

    # Use compact weekday labels on the x-axis.
    df_season_households.index = [idx.split()[0] for idx in df_season_households.index]
    season_appliance_percentage_df.index = [
        idx.split()[0]
        for idx in season_appliance_percentage_df.index
    ]
    df_season_households_aggregated.index = (
        df_season_households_aggregated.index
        .get_level_values(1)
        .str.slice(0, 3)
    )



    # --------------------------------
    #   Sec 2.9 -- 3.0) Prepare colors and appliance daily profiles
    # --------------------------------

    # Assign one color to each appliance in the ordered appliance list.
    n_appliances = len(appliances_ordered)
    appliance_colormap = cm.get_cmap('viridis', n_appliances)
    color_by_appliance = {
        appliance: appliance_colormap(i)
        for i, appliance in enumerate(appliances_ordered)
    }

    def rgba_to_hex(rgba):
        """Convert an RGBA tuple with values in [0, 1] to a hexadecimal color."""
        return '#{:02x}{:02x}{:02x}'.format(
            int(rgba[0] * 255),
            int(rgba[1] * 255),
            int(rgba[2] * 255)
        )

    color_by_appliance_hex = {
        appliance: rgba_to_hex(color_by_appliance[appliance])
        for appliance in appliances_ordered
    }

    # Add support columns for day category and time of day.
    all_appliances = appliance_timeseries_df.columns.tolist()
    used_appliances = [
        appliance
        for appliance in appliances_ordered
        if appliance in all_appliances
    ]

    appliance_timeseries_with_time_df = appliance_timeseries_df.copy()
    appliance_timeseries_with_time_df['dayofweek'] = appliance_timeseries_with_time_df.index.dayofweek
    appliance_timeseries_with_time_df['day_category'] = appliance_timeseries_with_time_df['dayofweek'].apply(
        lambda x: 'working_day' if x < 5 else ('saturday' if x == 5 else 'sunday')
    )
    appliance_timeseries_with_time_df['time'] = appliance_timeseries_with_time_df.index.strftime('%H:%M')

    # Build one representative daily appliance profile for each day category.
    appliance_profiles_by_day_category = {}
    for day_category in day_categories:
        category_df = appliance_timeseries_with_time_df[
            appliance_timeseries_with_time_df['day_category'] == day_category
        ]
        appliance_profiles_by_day_category[day_category] = category_df.groupby('time')[used_appliances].sum()

    legend_columns_for_appliances = max(1, math.ceil(len(used_appliances) / 20))



    # --------------------------------
    #   Sec 2.9 -- 4.0) Plot daily household consumption by household type
    # --------------------------------

    sorted_households = sorted(df_season_households.columns, key=get_type_order)
    df_daily_household_plot = df_season_households[sorted_households]

    # Group households by type and assign a color gradient within each group.
    households_by_type = {}
    for household_id in sorted_households:
        household_type = extract_household_type(household_id)
        households_by_type.setdefault(household_type, []).append(household_id)

    color_by_household = {}
    for household_type, household_list in households_by_type.items():
        base_color = color_by_household_type.get(household_type, '#999999')
        gradient_colors = get_gradation_colors(base_color, len(household_list))
        for household_id, household_color in zip(household_list, gradient_colors):
            color_by_household[household_id] = household_color

    household_color_list = [
        color_by_household[household_id]
        for household_id in sorted_households
    ]

    ax = df_daily_household_plot.plot(
        kind='bar',
        stacked=True,
        color=household_color_list,
        figsize=(12, 6)
    )
    ax.set_xlabel('Day of week')
    ax.set_ylabel('Energy consumption (kWh)')
    ax.set_title(
        f'Daily consumption for {num_households} households - {selected_season.title()}'
    )

    household_legend_columns = max(1, math.ceil(len(sorted_households) / 20))
    ax.legend(
        title='Households',
        ncol=household_legend_columns,
        bbox_to_anchor=(1.05, 1),
        loc='upper left'
    )
    plt.tight_layout()

    filename = os.path.join(
        output_pictures_folder,
        f'consumption_daily_{num_households}_{selected_season}_{selected_region}_{induction_share_pct}%I.png'
    )
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()



    # --------------------------------
    #   Sec 2.9 -- 5.0) Plot stacked appliance-consumption profiles by day category
    # --------------------------------

    appliances_used_for_stacked_plot = [
        appliance
        for appliance in appliances_ordered
        if appliance in used_appliances
    ]
    stacked_appliance_colors = [
        color_by_appliance.get(appliance)
        for appliance in appliances_used_for_stacked_plot
    ]
    stacked_appliance_labels = [
        appliance_english_dict.get(appliance, appliance)
        for appliance in appliances_used_for_stacked_plot
    ]

    for day_category in day_categories:
        ax = appliance_profiles_by_day_category[day_category][appliances_used_for_stacked_plot].plot(
            kind='bar',
            stacked=True,
            width=0.85,
            figsize=(14, 6),
            color=stacked_appliance_colors
        )
        ax.set_xlabel('Hour of the day')
        ax.set_ylabel('Energy consumption (kWh)')
        ax.set_title(
            f'Stacked appliance consumption - {day_category_english_dict[day_category]}\n'
            f'{num_households} households - {season_english_dict[selected_season]}'
        )

        ax.legend(
            stacked_appliance_labels,
            title='Appliances',
            ncol=legend_columns_for_appliances,
            bbox_to_anchor=(1.05, 1),
            loc='upper left'
        )

        x_labels = appliance_profiles_by_day_category[day_category].index.tolist()
        n_ticks = 24
        tick_positions = np.linspace(0, len(x_labels) - 1, n_ticks, dtype=int)
        tick_labels = [x_labels[i] for i in tick_positions]
        plt.xticks(tick_positions, tick_labels, rotation=45)
        plt.tight_layout()

        filename = os.path.join(
            output_pictures_folder,
            f'consumption_All_appl_{day_category}_{num_households}_households_{selected_season}_{selected_region}_{induction_share_pct}%I.png'
        )
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()



    # --------------------------------
    #   Sec 2.9 -- 6.0) Plot hourly profiles for selected appliances
    # --------------------------------

    appliances_of_interest = [
        appliance
        for appliance in all_appliances
        if appliance in allowed_appliances
    ]

    for appliance in tqdm(
        appliances_of_interest,
        desc='Creating hourly consumption plots for selected appliances'
    ):
        for day_category in day_categories:
            profile_df = appliance_profiles_by_day_category[day_category]
            x_positions = np.arange(len(profile_df.index))

            plt.figure(figsize=(10, 6))
            plt.bar(
                x_positions,
                profile_df[appliance],
                width=0.8,
                color=color_by_appliance.get(appliance, '#333333'),
                edgecolor='white',
                linewidth=0.5
            )

            plt.xlabel('Hour of the day')
            plt.ylabel('Energy consumption (kWh)')
            plt.title(
                f'Hourly consumption profile - {appliance_english_dict.get(appliance, appliance)} '
                f'({day_category_english_dict[day_category]} in {season_english_dict[selected_season]})\n'
                f'{num_households} households, zone: {selected_region}'
            )

            x_labels = profile_df.index.tolist()
            n_ticks = 24
            tick_positions = np.linspace(0, len(x_labels) - 1, n_ticks, dtype=int)
            tick_labels = [x_labels[i] for i in tick_positions]
            plt.xticks(tick_positions, tick_labels, rotation=45)
            plt.grid(True)
            plt.tight_layout()

            filename = os.path.join(
                appliance_pictures_output_folder,
                f"cons_{appliance.replace(' ', '_')}_{day_category}_{num_households}_households_{selected_season}_{selected_region}.png"
            )
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()



    # --------------------------------
    #   Sec 2.9 -- 7.0) Plot consumption shares by household category
    # --------------------------------

    n_household_category_colors = max(1, len(df_season_households_aggregated.columns))
    household_category_colors = [
        cm.plasma(i / n_household_category_colors)
        for i in range(n_household_category_colors)
    ]

    ax = df_season_households_aggregated.plot(
        kind='bar',
        stacked=True,
        color=household_category_colors,
        figsize=(10, 6)
    )
    ax.set_ylabel('Share (%)')
    ax.set_title(
        f'Consumption share by household category\n'
        f'Week in {season_english_dict[selected_season]} - {induction_share_pct}% induction hobs'
    )
    ax.legend(
        title='Household category',
        bbox_to_anchor=(1.05, 1),
        loc='upper left'
    )
    plt.tight_layout()

    filename = os.path.join(
        output_pictures_folder,
        f'consumption_percentage_{num_households}_households_{selected_season}.png'
    )
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()



    # --------------------------------
    #   Sec 2.9 -- 8.0) Plot annual appliance-consumption dispersion
    # --------------------------------

    # Reset the index if the appliance column was saved as the Excel index.
    if appliance_dispersion_by_household_df.index.name == 'Appliance':
        appliance_dispersion_by_household_df = appliance_dispersion_by_household_df.reset_index()

    grouped_appliances = appliance_dispersion_by_household_df.groupby('Appliance')['annual_kWh']
    max_values_by_appliance = grouped_appliances.max()
    mean_values_by_appliance = grouped_appliances.mean()

    # Normalize annual consumption using the maximum value of each appliance.
    appliance_dispersion_by_household_df['normalized'] = appliance_dispersion_by_household_df.apply(
        lambda row: row['annual_kWh'] / max_values_by_appliance[row['Appliance']],
        axis=1
    )

    # Sort appliances by increasing average annual consumption.
    appliance_labels = list(mean_values_by_appliance.sort_values().index)
    x_positions = np.arange(len(appliance_labels))

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Scatter plot of normalized household-level values.
    for i, appliance in enumerate(appliance_labels):
        subset = appliance_dispersion_by_household_df[
            appliance_dispersion_by_household_df['Appliance'] == appliance
        ]
        jitter = (np.random.rand(len(subset)) - 0.5) * 0.2
        ax1.scatter(
            x_positions[i] + jitter,
            subset['normalized'],
            alpha=0.7,
            edgecolor='k',
            c=color_by_appliance_hex.get(appliance, '#333333'),
            label='_nolegend_'
        )

    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(appliance_labels, rotation=45, ha='right')
    ax1.set_xlabel('Appliance')
    ax1.set_ylabel('Normalized annual-consumption distribution (0-1)')

    # Show annual average values on the secondary y-axis.
    ax2 = ax1.twinx()
    annual_means = [mean_values_by_appliance[appliance] for appliance in appliance_labels]
    ax2.bar(
        x_positions,
        annual_means,
        width=0.85,
        color='gray',
        alpha=0.3,
        label='Annual mean kWh'
    )
    ax2.set_ylabel('Average annual consumption (kWh)')
    ax2.legend(loc='upper left')

    plt.title(
        f'Consumption dispersion and annual averages by appliance\n'
        f'{num_households} households - {selected_region} - {induction_share_pct}% induction hobs'
    )
    plt.tight_layout()

    filename = os.path.join(
        output_pictures_folder,
        f'dispersion_appl_{num_households}_households_{selected_region}_{induction_share_pct}%I.png'
    )
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()






# ----------------------- OLD VERSION --------------------------------------------

# if RUN_APPLIANCE_PLOTS:
    
#     # ----------------- DAILY HOUSEHOLD summary DataFrame ------------------
#     household_summary_path = os.path.join(
#         consumption_output_xls_folder,
#         f'{num_households}_households_{selected_region}_summary_households_base.xlsx'
#     )

#     household_kwh_df = pd.read_excel(household_summary_path, sheet_name='kWh el', index_col=0)
#     household_percentage_df = pd.read_excel(household_summary_path, sheet_name='percentages', index_col=0)

#     # ---------------- Dataframe RIEPILOGO NUCLEI AGGREGATI ----------
#     aggregated_household_categories_df = pd.read_excel(os.path.join(
#         consumption_output_xls_folder,
#         f'{num_households}_households_{selected_region}_percentuali_category_household_base.xlsx'),
#         sheet_name= 'by_day', index_col=[0,1])

#     # ----------------- Dataframe TIMESERIES ELETTRODOMESTICI ------------------
#     appliance_timeseries_df = pd.read_excel(os.path.join(
#         consumption_output_xls_folder,
#         f'{num_households}_households_{selected_region}_ts_kWh_appliances_by_household.xlsx'), 
#         sheet_name= selected_season, 
#         index_col=0,
#         parse_dates=True
#         )

#     # ---------------- Dataframe RIEPILOGO ELETTRODOMESTICI GIORNALIERI ----------
#     appliance_summary_path = os.path.join(
#         consumption_output_xls_folder,
#         f'{num_households}_households_{selected_region}_summary_appliances.xlsx'
#     )

#     appliance_kwh_df = pd.read_excel(appliance_summary_path, sheet_name='kWh el', index_col=0)
#     appliance_percentage_df = pd.read_excel(appliance_summary_path, sheet_name='percentages', index_col=0)

#     # ---------------- Dataframe CONSUMI ELETTRODOMESTICI MIN/MAX/MEAN ANNO ---------
#     appliance_dispersion_path = os.path.join(
#         consumption_output_xls_folder,
#         f'{num_households}_households_{selected_region}_cons_appl_medio_summary.xlsx'
#     )

#     appliance_dispersion_by_household_df = pd.read_excel(appliance_dispersion_path, sheet_name='by_household', index_col=0)
#     # df_appl_dispers_tot = pd.read_excel(cons_appl_dispersione, sheet_name='Appliance_consumption', index_col=0)



#     # --------------- Add selected-season filters ----------------------

#     # Filter: day + season abbreviation, e.g. 'wed Aut'
#     abbr = season_abbr_dict[selected_season]
        
#     pattern = '|'.join(f'^{d} {abbr}$' for d in day_order)


#     # 0) Create the Boolean mask (filling NaN with False if needed)
#     mask = aggregated_household_categories_df.index.to_series().str.match(pattern).fillna(False)

#     # 1) Apply the mask with .loc
#     # Extract the second level and create the mask
#     lvl1 = pd.Series(aggregated_household_categories_df.index.get_level_values(1), index=aggregated_household_categories_df.index)
#     mask = lvl1.str.match(pattern).fillna(False)

#     df_season_households_aggregated = aggregated_household_categories_df.loc[mask]

#     # 2) Select ONLY daily rows for this season
#     df_season_households = household_kwh_df[household_kwh_df.index.str.match(pattern)]
#     # season_household_percentage_df = household_percentage_df[household_percentage_df.index.str.match(pattern)]
#     # df_season_appl = appliance_kwh_df[appliance_kwh_df.index.str.match(pattern)]
#     season_appliance_percentage_df = appliance_percentage_df[appliance_percentage_df.index.str.match(pattern)]

#     # 3) Reorder the rows according to the Wednesday-Tuesday order
#     ordered_idx = [f"{d} {abbr}" for d in day_order]
#     df_season_households = df_season_households.reindex(ordered_idx)
#     # season_household_percentage_df = season_household_percentage_df.reindex(ordered_idx)
#     # df_season_appl = df_season_appl.reindex(ordered_idx)
#     season_appliance_percentage_df = season_appliance_percentage_df.reindex(ordered_idx)


#     df_season_households_aggregated = df_season_households_aggregated.reindex(ordered_idx, level=1)

#     # 3) Extract only the first 3 characters for x-axis labels
#     df_season_households.index = [idx.split()[0] for idx in df_season_households.index]
#     # season_household_percentage_df.index = [idx.split()[0] for idx in season_household_percentage_df.index]
#     # df_season_appl.index = [idx.split()[0] for idx in df_season_appl.index]
#     season_appliance_percentage_df.index = [idx.split()[0] for idx in season_appliance_percentage_df.index]


#     # df_season_households_aggregated.index = [idx.split()[0] for idx in df_season_households_aggregated.index]
#     df_season_households_aggregated.index = df_season_households_aggregated.index.get_level_values(1).str.slice(0,3)


#     n_appliances = len(appliances_ordered)

#     # Choose a colormap (e.g., "plasma") and generate num_appliances colors distributed along the scale
#     cmap = cm.get_cmap("viridis", n_appliances)

#     # Create a mapping: each appliance in the ordered list is associated with the corresponding color
#     color_by_appliance = {appliance: cmap(i) for i, appliance in enumerate(appliances_ordered)}

    
#     # To use colors in hexadecimal format (e.g., "#FF5733"), convert them as follows:
#     def rgba_to_hex(rgba):
#         # rgba is a tuple (r, g, b, a) with values in [0, 1]
#         return '#{:02x}{:02x}{:02x}'.format(
#             int(rgba[0]*255),
#             int(rgba[1]*255),
#             int(rgba[2]*255)
#         )

#     color_by_appliance_hex = {appl: rgba_to_hex(color_by_appliance[appl]) for appl in appliances_ordered}

#     #---- Add day information to appliance time series -------
#     appliances_all = appliance_timeseries_df.columns.tolist()
#     used_appliances = [appl for appl in appliances_ordered if appl in appliances_all]

#     # Copy the DataFrame per evitare di modificare quello originale
#     df_el = appliance_timeseries_df.copy()

#     # ——— Add support columns for day and time ———
#     df_el['dayofweek'] = df_el.index.dayofweek  # 0 = Mon, ..., 6 = Sun
#     df_el['day_category'] = df_el['dayofweek'].apply(lambda x: 'working_day' if x < 5 else ('saturday' if x == 5 else 'sunday'))

#     # Extract the time (hours and minutes) as a string, e.g., "08:10"
#     df_el['time'] = df_el.index.strftime("%H:%M")

#     # Define day categories
#     day_categories = ['working_day', 'saturday', 'sunday']
#     day_category_english_dict = {
#         'working_day': 'Working day',
#         'saturday': 'Saturday',
#         'sunday': 'Sunday'
#         }


#     num_items = len(used_appliances)  # ad esempio: len(sorted_households)
#     num_columns = math.ceil(num_items / 20)  # If num_items=44, then num_columns = math.ceil(44/20) = 3


#     # For each category, group by time and calculate consumption.
#     profiles = {}
#     for cat in day_categories:
#         df_cat = df_el[df_el['day_category'] == cat]
#         time_profile = df_cat.groupby('time')[used_appliances].sum()  # or .mean() se preferisci
#         profiles[cat] = time_profile


#     # ----------- PLOT No. 1: DAILY HOUSEHOLD CONSUMPTION DURING THE WEEK (ordered by household type) ------------------------------------
#     # Sort columns by household type
#     sorted_households = sorted(df_season_households.columns, key=get_type_order)
#     df_plot = df_season_households[sorted_households]
    
#     # Group households by type and assign gradients
#     groups = {}
#     for household_id in sorted_households:
#         typ = extract_household_type(household_id)
#         groups.setdefault(typ, []).append(household_id)
        
#     household_colors = {}
#     for typ, lst in groups.items():
#         base = color_by_household_type.get(typ, '#999999')
#         grad = get_gradation_colors(base, len(lst))
#         for household_id, col in zip(lst, grad):
#             household_colors[household_id] = col
#     colors_list = [household_colors[n] for n in sorted_households]

#     # Plot
#     ax = df_plot.plot(kind='bar', stacked=True, color=colors_list, figsize=(12,6))
#     ax.set_xlabel('Day of week')
#     ax.set_ylabel('Power consumption (kWh)')
#     ax.set_title(f'Daily consumption for {num_households} households - {selected_season.title()}')# \n {share_induzione_100}% Induction hob')
#     # Legend in multiple columns
#     ticks = math.ceil(len(sorted_households)/20)
#     ax.legend(title='Households', ncol=ticks, bbox_to_anchor=(1.05,1), loc='upper left')
#     plt.tight_layout()
#     # Savetaggio
#     dest = os.path.join(output_pictures_folder, f"consumption_daily_{num_households}_{selected_season}_{selected_region}_{induction_share_pct}%I.png")
#     plt.savefig(dest, dpi=300, bbox_inches='tight')
#     plt.close()
#     plt.show()
#     #------------------------------------------------------------------------------------------------------------------------------------------

#     #------------- GRAFICO  N° 2: PROFILI CONSUMI ELETTRODOMESTICI messi insieme --------------------------------------------------------------
#     # Stacked bar chart impilate per i consumption degli appliances
#     for cat in day_categories:
#         plt.figure(figsize=(14, 6))
#         appliances_used = [appl for appl in appliances_ordered if appl in used_appliances]
#         colors_appl = [color_by_appliance.get(appl) for appl in appliances_used]
        
#         ax = profiles[cat][appliances_used].plot(kind='bar', stacked=True, width=0.85, figsize=(14, 6), color=colors_appl) # or kind='area' per aree in pila
#         plt.xlabel("Hour of the day ")
#         plt.ylabel("Power consumption (kWh)")
#         plt.title(f"Stacked consumption (appliances) - {day_category_english_dict[cat]} \n {num_households} households - {season_english_dict[selected_season]} ")#- {share_induzione_100}% Induction hob")
#         legend_labels = [appliance_english_dict.get(appl, appl) for appl in appliances_used]

#         plt.legend(
#             legend_labels,
#             title="Appliances",
#             ncol=num_columns, 
#             bbox_to_anchor=(1.05, 1), 
#             loc='upper left')
#         plt.tight_layout()
        
#         # Reduce the number of ticks on the x-axis
#         x_labels = profiles[cat].index.tolist()
#         n_ticks = 24  # define qui il numero desiderato di tick
#         tick_positions = np.linspace(0, len(x_labels)-1, n_ticks, dtype=int)
#         tick_labels = [x_labels[i] for i in tick_positions]
#         plt.xticks(tick_positions, tick_labels, rotation=45)
        
        
#         # plt.tight_layout()
#         filename = os.path.join(output_pictures_folder, f"consumption_All_appl_{cat}_{num_households}_households_{selected_season}_{selected_region}_{induction_share_pct}%I.png")
#         plt.savefig(filename, dpi=300, bbox_inches="tight")
#         plt.close()
#     #------------------------------------------------------------------------------------------------------------------------------------------

#     # --------------- GRAFICO  N° 3: PROFILI INDIVIDUALI SINGOLI ELETTRODOMESTICI (solo per i più interessanti, guarda definizione parametri)
#     appliances_of_interest = [appl for appl in appliances_all if appl in allowed_appliances]
#     num_columns = math.ceil(len(appliances_of_interest)/4)  # legende

#     # Individual plots for each appliance and each day category:
#     for appliance in tqdm(appliances_of_interest, desc = "Creating hourly consumption plots for the different appliances"):
#         for cat in day_categories:
            
#             # ax = profiles[cat][appliances_used].plot(kind='bar', stacked=True, width=0.85, figsize=(10, 6), color=colors_appl)
#             plt.figure(figsize=(10, 6))
            
#             # Create a numeric sequence for the x-axis based on the number of timesteps
#             x_positions = np.arange(len(profiles[cat].index))
            
#             # Plot the bar chart:
#             plt.bar(
#                 x_positions, 
#                 profiles[cat][appliance], 
#                 width=0.8,  # adjust the width as desired
#                 color=color_by_appliance.get(appliance, "#333333"),
#                 edgecolor='white',  # white edge
#                 linewidth=0.5       # reduced edge width
#             )
            

#             plt.xlabel("Hour of the day")
#             plt.ylabel("Power consumption (kWh)")
#             plt.title(f"Hourly consumption profile {appliance_english_dict[appliance]} ({day_category_english_dict[cat]} in {season_english_dict[selected_season]}) \n {num_households} households, Zone: {selected_region}")

#             # Set x-axis ticks to show fewer labels
#             x_labels = profiles[cat].index.tolist()
#             n_ticks = 24  # for example, 10 ticks on the axis
#             tick_positions = np.linspace(0, len(x_labels)-1, n_ticks, dtype=int)
#             tick_labels = [x_labels[i] for i in tick_positions]
#             plt.xticks(tick_positions, tick_labels, rotation=45)
            
#             plt.grid(True)
#             # Save the plot (adapt the path and file name if needed)
            
#             # If tight_layout generates warnings, try adjusting the margins manually:
#             plt.tight_layout()  # or usa: plt.subplots_adjust(bottom=0.15, top=0.9)
                        
#             filename = os.path.join(appliance_pictures_output_folder, f"cons_{appliance.replace(' ', '_')}_{cat}_{num_households}_households_{selected_season}_{selected_region}.png")
#             plt.savefig(filename)       
#             plt.close()
#     #------------------------------------------------------------------------------------------------------------------------------------------
    

#     #-------------- No. 4: PERCENTAGE WEIGHTS OF CONSUMPTION BY HOUSEHOLD TYPE AND APPLIANCE ------------------------------------------------------
#     # ----Grafico ripartizioni percentuali classi households----
#     n_colors_n = len(df_season_households_aggregated.columns)

#     # Generate a color list using the plasma colormap
#     household_colors = [cm.plasma(i / n_colors_n) for i in range(n_colors_n)]
#     plt.figure(figsize=(10, 6))
#     ax = df_season_households_aggregated.plot(kind='bar', stacked=True, color=household_colors)

#     ax.set_ylabel("Share (%)")
#     ax.set_title(f"Share of consumption per type of household class \n Week in {season_english_dict[selected_season]} - {induction_share_pct}% Induction hob")
#     # Place the legend outside the figure
#     plt.legend(title="Type of household class", bbox_to_anchor=(1.05, 1), loc='upper left')
#     plt.tight_layout()
#     filename = os.path.join(output_pictures_folder, f"consumption_percentage_{num_households}_households_{selected_season}.png")
#     plt.savefig(filename)  

#     plt.show()
#     #------------------------------------------------------------------------------------------------------------------------------------------
    
#     #---------------- GRAFICO  N° 5: DISPERSIONE CONSUMI ELETTRODOMESTICI per type di appl ------------------------------------------
#     # Normalize annual consumption for each category so it can be shown in the same plot
#     if appliance_dispersion_by_household_df.index.name == 'Appliance':
#         appliance_dispersion_by_household_df = appliance_dispersion_by_household_df.reset_index()

#     grouped = appliance_dispersion_by_household_df.groupby('Appliance')['annual_kWh']
#     max_vals = grouped.max()
#     mean_vals = grouped.mean()
#     appliance_dispersion_by_household_df['normalized'] = appliance_dispersion_by_household_df.apply(lambda row: row['annual_kWh']/max_vals[row['Appliance']], axis=1)


#     # ——— 2) 0-1 normalization using the maximum of each appliance ———
#     appliance_dispersion_by_household_df['normalized'] = appliance_dispersion_by_household_df.apply(
#         lambda r: r['annual_kWh'] / max_vals[r['Appliance']], axis=1
#     )

#     # ——— 3) X-axis preparation (one position for each appliance) ———
#     appliance_labels = list(mean_vals.sort_values().index) # sort by increasing average annual consumption

#     x_pos    = np.arange(len(appliance_labels))

#     # ——— 4) Scatter plot of normalized values ———
#     fig, ax1 = plt.subplots(figsize=(12, 6))

#     for i, appl in enumerate(appliance_labels):
#         subset = appliance_dispersion_by_household_df[
#             appliance_dispersion_by_household_df['Appliance'] == appl
#         ]
#         # slight jitter along x to separate the points
#         jitter = (np.random.rand(len(subset)) - 0.5) * 0.2
#         ax1.scatter(
#             x_pos[i] + jitter,
#             subset['normalized'],
#             alpha=0.7,
#             edgecolor='k',
#             c=color_by_appliance_hex.get(appl, "#333333"),
#             label='_nolegend_'
#         )

#     ax1.set_xticks(x_pos)
#     ax1.set_xticklabels(appliance_labels, rotation=45, ha='right')
#     ax1.set_xlabel("Appliance")
#     ax1.set_ylabel("Distribution normalised annual consumption (0–1)")

#     # ——— 5) Annual-average bars on the secondary axis ———
#     ax2 = ax1.twinx()
#     means = [mean_vals[appl] for appl in appliance_labels]
#     ax2.bar(
#         x_pos,
#         means,
#         width=0.85,
#         color='gray',
#         alpha=0.3,
#         label='Annual mean kWh'
#     )
#     ax2.set_ylabel("Average annual consumption (kWh)")

#     # ——— 6) Legend and saving ———
#     ax2.legend(loc='upper left')
#     plt.title(f"Consumption dispersion and annual averages per appliance\n" #  Consumption dispersion and annual averages by appliance
#             f"{num_households} households – {selected_region} - {induction_share_pct}% Induction hobs")
#     plt.tight_layout()

#     filename = os.path.join(
#         output_pictures_folder,
#         f"dispersion_appl_{num_households}_households_{selected_region}_{induction_share_pct}%I.png"
#     )
#     plt.savefig(filename, dpi=300, bbox_inches='tight')
#     plt.close()
#     #------------------------------------------------------------------------------------------------------------------------------------------


C:\Users\Francesco\AppData\Local\Temp\ipykernel_13496\282983626.py:142: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  appliance_colormap = cm.get_cmap('viridis', n_appliances)
C:\Users\Francesco\AppData\Local\Temp\ipykernel_13496\282983626.py:232: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
Creating hourly consumption plots for selected appliances: 100%|██████████| 2/2 [00:04<00:00,  2.48s/it]
C:\Users\Francesco\AppData\Local\Temp\ipykernel_13496\282983626.py:404: RuntimeWarning: invalid value encountered in scalar divide
  lambda row: row['annual_kWh'] / max_values_by_appliance[row['Appliance']],


 ## 2.XXX Conclusione

In [14]:
# Ottenere l'time di fine
end_time = datetime.datetime.now()
print(f"End time : {end_time}")

# Calculate execution time
execution_time = end_time - start_time
print(f"Execution time : {execution_time}")

End time : 2026-07-10 10:32:32.202172
Execution time : 0:37:21.679831
